# Aumentando el realismo en modelos de AgentPy: Flocking Behavior

TC2008B · Tec de Monterrey

**Integrantes:** Carlos Eduardo Arias Capetillo · Ilan David Narváez Martínez · Joel Martínez Escalera · Mario Giovanni González López · Oswaldo García Torres

**Cómo ejecutar:** `pip install -r requirements.txt` y luego *Run All*. Todas las corridas usan una semilla fija (ver sección 0).

**Índice**
0. Configuración
1. Parte I: modelo base y preguntas guía
2. Parte II: mejoras candidatas y priorización
3. Parte III: modelo extendido
4. Calibración
5. Recolección de datos (KPIs)
6. Experimentos
7. Validación
8. Conclusiones y limitaciones
9. Referencias

## 0. Configuración

In [ ]:
# Model design
import agentpy as ap
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import IPython

# Data analysis
import pandas as pd

## 1. Parte I: modelo base y preguntas guía


### 1.0 Modelo base (sin modificar)

**Sobre el modelo.** Es el modelo de *boids* de Craig Reynolds (1986): cada agente ajusta su movimiento con tres reglas locales basadas en sus vecinos cercanos.
- **Separación:** alejarse de vecinos demasiado cercanos.
- **Alineación:** orientarse hacia la dirección promedio de los vecinos.
- **Cohesión:** moverse hacia la posición promedio de los vecinos.

La implementación de AgentPy sigue el *Boids Pseudocode* de Conrad Parker y añade una cuarta regla para evitar los bordes del espacio. Funciona en 2 o 3 dimensiones.

**Estructura.** Hay dos clases: una para los agentes (`Boid`) y otra para el modelo (`BoidsModel`). Cada agente empieza con posición y velocidad aleatorias (arreglos de numpy). La posición vive en el espacio, al que el agente accede con `pos` y `neighbors()`. `update_velocity()` y `update_position()` están separados para que **todos** los agentes calculen su nueva velocidad antes de que cualquiera se mueva.

#### Función auxiliar

In [ ]:
def normalize(v):
    """ Normalize a vector to length 1. """
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm

#### Agente

In [ ]:
class Boid(ap.Agent):
    """ An agent with a position and velocity in a continuous space,
    who follows Craig Reynolds three rules of flocking behavior;
    plus a fourth rule to avoid the edges of the simulation space. """

    def setup(self):

        self.velocity = normalize(
            self.model.nprandom.random(self.p.ndim) - 0.5)

    def setup_pos(self, space):

        self.space = space
        self.neighbors = space.neighbors
        self.pos = space.positions[self]

    def update_velocity(self):

        pos = self.pos
        ndim = self.p.ndim

        # Rule 1 - Cohesion
        nbs = self.neighbors(self, distance=self.p.outer_radius)
        nbs_len = len(nbs)
        nbs_pos_array = np.array(nbs.pos)
        nbs_vec_array = np.array(nbs.velocity)
        if nbs_len > 0:
            center = np.sum(nbs_pos_array, 0) / nbs_len
            v1 = (center - pos) * self.p.cohesion_strength
        else:
            v1 = np.zeros(ndim)

        # Rule 2 - Seperation
        v2 = np.zeros(ndim)
        for nb in self.neighbors(self, distance=self.p.inner_radius):
            v2 -= nb.pos - pos
        v2 *= self.p.seperation_strength

        # Rule 3 - Alignment
        if nbs_len > 0:
            average_v = np.sum(nbs_vec_array, 0) / nbs_len
            v3 = (average_v - self.velocity) * self.p.alignment_strength
        else:
            v3 = np.zeros(ndim)

        # Rule 4 - Borders
        v4 = np.zeros(ndim)
        d = self.p.border_distance
        s = self.p.border_strength
        for i in range(ndim):
            if pos[i] < d:
                v4[i] += s
            elif pos[i] > self.space.shape[i] - d:
                v4[i] -= s

        # Update velocity
        self.velocity += v1 + v2 + v3 + v4
        self.velocity = normalize(self.velocity)

    def update_position(self):

        self.space.move_by(self, self.velocity)

#### Modelo

In [ ]:
class BoidsModel(ap.Model):
    """
    An agent-based model of animals' flocking behavior,
    based on Craig Reynolds' Boids Model [1]
    and Conrad Parkers' Boids Pseudocode [2].

    [1] http://www.red3d.com/cwr/boids/
    [2] http://www.vergenet.net/~conrad/boids/pseudocode.html
    """

    def setup(self):
        """ Initializes the agents and network of the model. """

        self.space = ap.Space(self, shape=[self.p.size]*self.p.ndim)
        self.agents = ap.AgentList(self, self.p.population, Boid)
        self.space.add_agents(self.agents, random=True)
        self.agents.setup_pos(self.space)

    def step(self):
        """ Defines the models' events per simulation step. """

        self.agents.update_velocity()   # Adjust direction
        self.agents.update_position()   # Move into new direction

#### Visualización

`animation_plot` recibe el modelo y los parámetros, y crea una animación con `ap.animate()`.

In [ ]:
def animation_plot_single(m, ax):
    ndim = m.p.ndim
    ax.set_title(f"Boids Flocking Model {ndim}D t={m.t}")
    pos = m.space.positions.values()
    pos = np.array(list(pos)).T  # Transform
    ax.scatter(*pos, s=1, c='black')
    ax.set_xlim(0, m.p.size)
    ax.set_ylim(0, m.p.size)
    if ndim == 3:
        ax.set_zlim(0, m.p.size)
    ax.set_axis_off()

def animation_plot(m, p):
    projection = '3d' if p['ndim'] == 3 else None
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111, projection=projection)
    animation = ap.animate(m(p), fig, ax, animation_plot_single)
    return IPython.display.HTML(animation.to_jshtml(fps=20))

#### Simulación 2D

Los parámetros se definen en un diccionario.

In [ ]:
parameters2D = {
    'size': 50,
    'seed': 123,
    'steps': 200,
    'ndim': 2,
    'population': 200,
    'inner_radius': 3,
    'outer_radius': 10,
    'border_distance': 10,
    'cohesion_strength': 0.005,
    'seperation_strength': 0.1,
    'alignment_strength': 0.3,
    'border_strength': 0.5
}

In [ ]:
animation_plot(BoidsModel, parameters2D)

#### Simulación 3D

Lo mismo en tres dimensiones, con más agentes.

In [ ]:
new_parameters = {
    'ndim': 3,
    'population': 1000,
}

parameters3D = dict(parameters2D)
parameters3D.update(new_parameters)

animation_plot(BoidsModel, parameters3D)

#### Métricas de apoyo

Dos funciones para medir el estado de la bandada. Se usan en las respuestas de abajo y se reutilizarán como KPIs en la sección 5.
- **Polarización:** magnitud de la velocidad promedio. Como cada velocidad mide 1, vale 0 si cada agente va en una dirección distinta y 1 si todos van en la misma.
- **Distancia al vecino más cercano:** promedio, sobre todos los agentes, de la distancia a su vecino más próximo. Mide qué tan compacta está la bandada.

In [ ]:
def polarizacion(m):
    """ Magnitud de la velocidad promedio de los agentes (0 = desorden, 1 = alineación total). """
    return np.linalg.norm(np.mean([a.velocity for a in m.agents], axis=0))

def dist_vecino_mas_cercano(m):
    """ Distancia promedio de cada agente a su vecino más cercano. """
    pos = np.array(list(m.space.positions.values()))
    d = np.linalg.norm(pos[:, None] - pos[None], axis=-1)
    np.fill_diagonal(d, np.inf)
    return d.min(axis=1).mean()

### 1. Estructura del modelo

**(a) Componentes principales de un modelo AgentPy**

*Respuesta:* Un modelo de AgentPy se compone de:
- **Parámetros:** un diccionario (`parameters2D`) al que el modelo y los agentes acceden como `self.p`.
- **Modelo:** una subclase de `ap.Model` (`BoidsModel`) que define el ciclo de la simulación.
- **Agentes:** una subclase de `ap.Agent` (`Boid`), agrupados en una `ap.AgentList`.
- **Entorno:** aquí un `ap.Space` continuo.
- **Generadores aleatorios:** `model.random` (Python) y `model.nprandom` (numpy), sembrados con el parámetro `seed`.
- **Salida:** un `ap.DataDict` en `model.output` con la información, los parámetros y los datos registrados.

**(b) Cómo se definen y relacionan `ap.Model` y `ap.Agent`**

*Respuesta:* `BoidsModel` hereda de `ap.Model` y `Boid` de `ap.Agent`. El modelo **contiene** a los agentes y al espacio: en su `setup()` crea `self.space` y `self.agents = ap.AgentList(self, population, Boid)`. Cada agente recibe al crearse una referencia a su modelo (`self.model`), y a través de ella accede a los parámetros (`self.p`) y a los generadores aleatorios (`self.model.nprandom`). El modelo controla el tiempo y llama a los métodos de los agentes; los agentes no se llaman entre sí, solo leen el estado de sus vecinos a través del espacio.

**(c) Diferencia entre `setup()` y `step()`**

*Respuesta:* `setup()` corre **una sola vez**, en t = 0, y prepara la simulación: crea el espacio, los agentes y sus posiciones. `step()` corre **en cada paso** (t = 1, 2, …, `steps`) y define la dinámica: primero todos actualizan su velocidad y luego todos se mueven. El ciclo completo de `model.run()` es `setup → update → [t += 1 → step → update] × steps → end`. Además, `Boid` tiene su propio `setup()`, que AgentPy llama al crear cada agente y que le asigna una velocidad inicial.

### 2. Agentes

**(a) Cómo se crean y reciben atributos**

*Respuesta:* En tres pasos dentro de `BoidsModel.setup()`:
1. `ap.AgentList(self, self.p.population, Boid)` crea 200 agentes. Al crear cada uno se ejecuta `Boid.setup()`, que le asigna una velocidad aleatoria de magnitud 1.
2. `self.space.add_agents(self.agents, random=True)` les asigna una posición aleatoria en el espacio.
3. `self.agents.setup_pos(self.space)` llama ese método en cada agente para que guarde referencias al espacio, a la función de vecinos y a su posición.

Todos los agentes leen los mismos parámetros (`self.p`), así que no hay atributos individuales: solo difieren en posición y velocidad iniciales.

**(b) Variables o estados de cada agente**

*Respuesta:*
- `velocity`: vector de dimensión `ndim` y magnitud 1 (dirección de movimiento).
- `pos`: posición. No es una copia: es **el mismo arreglo** que guarda el espacio, así que se actualiza solo cuando el agente se mueve (verificado abajo).
- `space` y `neighbors`: referencias al espacio y a su función de búsqueda de vecinos.
- `id`: identificador que asigna AgentPy.

No tienen otro estado interno (energía, rol, memoria, objetivo).

**(c) Cómo perciben su entorno**

*Respuesta:* Con `self.neighbors(self, distance=r)`, que usa un árbol k-d para devolver **todos** los agentes dentro de un radio euclidiano `r`. Hay dos radios: `outer_radius = 10` para cohesión y alineación, e `inner_radius = 3` para separación. La percepción es de 360°, sin límite de vecinos y perfecta: cada agente conoce la posición y la velocidad exactas de sus vecinos, sin ruido. Además, cada agente "percibe" los bordes comparando su propia posición con el tamaño del espacio.

In [ ]:
m = BoidsModel(parameters2D)
m.sim_setup()  # Solo ejecuta setup(), sin avanzar pasos
a = m.agents[0]
print('Atributos propios del agente:', [k for k in vars(a) if not k.startswith('_')])
print('velocity =', a.velocity, '| magnitud =', np.linalg.norm(a.velocity))
print('pos =', a.pos)
print('¿pos es el mismo arreglo que guarda el espacio?', a.pos is m.space.positions[a])
print('Vecinos en outer_radius:', len(a.neighbors(a, distance=m.p.outer_radius)),
      '| en inner_radius:', len(a.neighbors(a, distance=m.p.inner_radius)))

### 3. Ambiente

**(a) Tipo de entorno**

*Respuesta:* Un `ap.Space`, es decir, un espacio **continuo** (las posiciones son números reales, no celdas). Su forma es `[size] * ndim`: un cuadrado de 50 × 50 en 2D o un cubo de 50 × 50 × 50 en 3D. No es toroidal (`torus=False` por defecto), así que los bordes son paredes y no se conectan entre sí.

**(b) Cómo se colocan los agentes**

*Respuesta:* `add_agents(..., random=True)` les da a cada uno una posición uniforme en [0, size) en cada eje, usando `model.random`. No hay agrupamiento inicial ni control de densidad: con 200 agentes en 2500 unidades², la densidad es de 0.08 agentes por unidad².

**(c) Reglas espaciales que afectan movimiento e interacción**

*Respuesta:*
- **Vecindad métrica:** solo interactúan los agentes dentro de un radio, sin importar cuántos sean.
- **Bordes suaves (regla 4):** si un agente está a menos de `border_distance = 10` de un borde, se suma `border_strength = 0.5` hacia adentro en ese eje.
- **Bordes duros:** si aun así un movimiento lo sacaría del espacio, AgentPy recorta la posición al límite (`[0, size]`).
- No hay obstáculos ni zonas con propiedades distintas.

Con `size = 50` y `border_distance = 10`, la zona libre de la regla de bordes es solo el cuadrado central de 30 × 30, así que **el 64% del área está bajo influencia de los bordes**. El borde es un factor dominante en este modelo.

In [ ]:
p = parameters2D
libre = (p['size'] - 2 * p['border_distance']) ** p['ndim']
print(f"Fracción del espacio bajo influencia de bordes: {1 - libre / p['size'] ** p['ndim']:.0%}")
print('Densidad inicial (agentes por unidad de área):', p['population'] / p['size'] ** p['ndim'])

### 4. Reglas de comportamiento

**(a) Cómo se codifica el comportamiento en `step()`**

*Respuesta:* `BoidsModel.step()` hace dos llamadas: `self.agents.update_velocity()` y después `self.agents.update_position()`. En `update_velocity()`, cada agente calcula cuatro vectores:
- **v1, cohesión:** `(centro de los vecinos − pos) × cohesion_strength`, con los vecinos en `outer_radius`.
- **v2, separación:** `−Σ (pos del vecino − pos) × seperation_strength`, con los vecinos en `inner_radius`.
- **v3, alineación:** `(velocidad promedio de los vecinos − velocidad propia) × alignment_strength`.
- **v4, bordes:** empuje de `border_strength` hacia adentro si está cerca de un borde.

Suma los cuatro a su velocidad y la **normaliza a magnitud 1**. Después, `update_position()` mueve al agente `velocity` unidades con `space.move_by`.

**(b) Reglas de decisión, procesos aleatorios o condicionales**

*Respuesta:* Solo hay condicionales simples: si no hay vecinos, la cohesión y la alineación valen cero; y la regla de bordes revisa eje por eje si el agente está cerca de una pared. No hay decisiones discretas, estados ni roles, y **la dinámica no tiene azar**: después de la inicialización, todo es determinista.

**(c) Cómo interactúan entre sí y con el ambiente**

*Respuesta:* Entre agentes, la interacción es **indirecta y local**: cada uno lee las posiciones y velocidades de sus vecinos y ajusta su propia dirección; no hay comunicación ni contacto. Con el ambiente, la única interacción son los bordes.

Dos observaciones:
- Como la velocidad se normaliza, **todos se mueven siempre a rapidez 1**; las reglas solo cambian la dirección.
- La separación es **proporcional a la distancia**: un vecino a 2.9 unidades empuja más que uno a 0.5. Es lo contrario de lo que se esperaría físicamente; otras implementaciones de boids usan una fuerza que crece al acercarse (por ejemplo, 1/d).

### 5. Agendamiento

**(a) Qué scheduler se usa**

*Respuesta:* AgentPy no tiene clases `RandomActivation` ni `SequentialActivation` (esas son de la librería Mesa). Aquí, llamar un método sobre una `AgentList` (`self.agents.update_velocity()`) lo ejecuta en cada agente **en el orden de la lista**, que es el orden de creación y siempre es el mismo. En términos de la pregunta, equivale a **activación secuencial fija**. El paso está dividido en dos fases: todos calculan su velocidad y, después, todos se mueven.

**(b) Cómo afecta el orden de activación a los resultados**

*Respuesta:*
- **En las posiciones, no afecta:** nadie se mueve hasta que todos calcularon su velocidad, así que cada agente ve las posiciones del mismo instante.
- **En las velocidades, sí afecta:** `velocity` se actualiza en el momento en que cada agente la calcula. Por eso, en la regla de alineación, un agente que se activa tarde ya ve las velocidades **nuevas** de los que se activaron antes, mientras que uno que se activa temprano ve las viejas.

En la prueba de abajo, con la misma semilla, el orden fijo y un orden barajado en cada paso terminan con posiciones distintas. Sin embargo, la polarización final promedio sobre 5 semillas es parecida (≈ 0.84 vs ≈ 0.86). Es decir, el orden cambia las trayectorias individuales, pero con 5 semillas no se nota en el comportamiento colectivo; habría que confirmarlo con ≥ 20 réplicas. Una actualización realmente sincrónica guardaría primero una copia de todas las velocidades.

In [ ]:
class BoidsModelOrdenAleatorio(BoidsModel):
    """ Igual que el modelo base, pero baraja el orden de activación en cada paso. """
    def step(self):
        self.agents.shuffle().update_velocity()
        self.agents.update_position()

def posiciones_finales(M, p):
    m = M(p)
    m.run(display=False)
    return np.array(list(m.space.positions.values())), polarizacion(m)

pos_fijo, _ = posiciones_finales(BoidsModel, parameters2D)
pos_aleatorio, _ = posiciones_finales(BoidsModelOrdenAleatorio, parameters2D)
print('Misma semilla, orden fijo vs barajado → posiciones idénticas:', np.array_equal(pos_fijo, pos_aleatorio))

for nombre, M in [('Orden fijo (base)', BoidsModel), ('Orden barajado', BoidsModelOrdenAleatorio)]:
    pols = [posiciones_finales(M, dict(parameters2D, seed=s))[1] for s in range(5)]
    print(f'{nombre}: polarización final en 5 semillas = {np.round(pols, 3)}, media = {np.mean(pols):.3f}')

### 6. Aleatoriedad y reproducibilidad

**(a) Cómo se introduce aleatoriedad**

*Respuesta:* Solo en las **condiciones iniciales**:
- **Posición:** uniforme en el espacio, con `model.random` (generador de Python).
- **Velocidad:** `normalize(nprandom.random(ndim) − 0.5)`, con `model.nprandom` (generador de numpy). Toma un punto uniforme en un cuadrado y lo normaliza, así que la dirección no es exactamente uniforme: favorece un poco las diagonales.

Después de t = 0 no hay más azar; la dinámica es determinista.

**(b) Cómo reproducir resultados con semilla fija**

*Respuesta:* Con el parámetro `'seed': 123`. Al iniciar, AgentPy crea `model.random` con esa semilla y siembra `model.nprandom` con un número derivado de él, así que una sola semilla controla ambos generadores. La prueba de abajo confirma que la misma semilla da exactamente las mismas posiciones finales y una semilla distinta no. Para réplicas, `ap.Experiment(..., randomize=True)` genera una semilla distinta por réplica a partir de la del `Sample`.

Para que esto se mantenga en las mejoras, **todo el azar debe venir de `self.model.random` o `self.model.nprandom`**; usar `np.random` o `random` directamente rompería la reproducibilidad.

In [ ]:
def posiciones(seed):
    m = BoidsModel(dict(parameters2D, seed=seed))
    m.run(display=False)
    return np.array(list(m.space.positions.values()))

print('Misma semilla (123 y 123) → resultados idénticos:', np.array_equal(posiciones(123), posiciones(123)))
print('Semilla distinta (123 y 7) → resultados idénticos:', np.array_equal(posiciones(123), posiciones(7)))

### 7. Colección de datos

**(a) Cómo recolecta datos el modelo**

*Respuesta:* El modelo base **no recolecta nada**: no llama a `record()` ni a `report()`, y no define `update()` ni `end()`. Su salida solo contiene información de la corrida, los parámetros y la semilla (ver abajo). AgentPy ofrece:
- `self.record('nombre', valor)` dentro de `update()`, para guardar una variable en cada paso. También funciona en agentes, por ejemplo `self.agents.record('velocity')`.
- `self.report('nombre', valor)` dentro de `end()`, para guardar un valor final de la corrida.

`model.run()` devuelve un `ap.DataDict` con `variables` (series de tiempo) y `reporters` (valores finales) como DataFrames de pandas.

**(b) Variables clave rastreadas en el tiempo**

*Respuesta:* Ninguna en el modelo base. Las que proponemos, porque miden directamente lo que producen las reglas:
- **Polarización:** efecto de la alineación.
- **Distancia al vecino más cercano:** equilibrio entre cohesión y separación.
- Candidatas adicionales: dispersión del grupo, número de grupos, fracción de agentes en la zona de bordes.

En la celda de abajo se registran las dos primeras con `record()`. La polarización sube de ~0.1 a ~0.9 y la distancia al vecino baja de ~1.8 a ~1.05.

**(c) Cómo se usan los datos para análisis o visualización**

*Respuesta:* En el ejemplo, solo para la animación: `ap.animate` lee `m.space.positions` en cada paso y dibuja los puntos. No hay análisis cuantitativo. Con `record()`, el `DataDict` se convierte en DataFrames para graficar series de tiempo, comparar configuraciones y calcular intervalos de confianza entre réplicas.

In [ ]:
m = BoidsModel(parameters2D)
salida = m.run(display=False)
print('Lo que guarda el modelo base:', list(salida.keys()))
print(salida.reporters)

class BoidsModelConDatos(BoidsModel):
    """ Modelo base + registro de dos métricas en cada paso. """
    def update(self):
        self.record('polarizacion', polarizacion(self))
        self.record('dist_vecino', dist_vecino_mas_cercano(self))

datos = BoidsModelConDatos(parameters2D).run(display=False)
serie = datos.variables.BoidsModelConDatos
ax = serie.plot(subplots=True, figsize=(7, 4), title='Modelo base, una corrida (seed = 123)')
plt.xlabel('Paso t')
plt.tight_layout()
plt.show()
print(serie.loc[[0, 50, 100, 200]].round(3))

### 8. Experimentos

**(a) Cómo ejecutar el modelo varias veces con parámetros distintos**

*Respuesta:* Hay dos formas:
- **Manual:** un ciclo que crea `BoidsModel(p).run()` para cada combinación de parámetros y semilla, y junta los resultados.
- **Con AgentPy:** se define el espacio de parámetros con `ap.Sample`, usando `ap.Values` (lista de valores), `ap.Range` o `ap.IntRange` (rangos), y se ejecuta con `ap.Experiment`.

**(b) Cómo ayuda `ap.Experiment` con barridos o réplicas**

*Respuesta:* `ap.Experiment(Modelo, sample, iterations=n, record=True, randomize=True)` corre cada combinación del `Sample` `n` veces con semillas distintas. Junta todo en un solo `DataDict`, identificando cada corrida por `sample_id` e `iteration`, y puede ejecutarse en paralelo con `n_jobs`. Para 20 réplicas por configuración basta con `iterations=20`.

La celda de abajo es una prueba pequeña que confirma que funciona con las versiones instaladas: dos valores de `alignment_strength`, 3 réplicas y 30 pasos. La polarización final media es mayor con más alineación (~0.27 vs ~0.11).

In [ ]:
class BoidsModelReporta(BoidsModel):
    def end(self):
        self.report('polarizacion_final', polarizacion(self))

muestra = ap.Sample(dict(parameters2D, steps=30, alignment_strength=ap.Values(0.1, 0.3)))
exp = ap.Experiment(BoidsModelReporta, muestra, iterations=3, randomize=True)
resultados = exp.run(display=False)
resultados.arrange_reporters().groupby('alignment_strength')['polarizacion_final'].agg(['mean', 'std', 'count'])

### 9. Validación e interpretación

**(a) ¿Los resultados hacen sentido frente a lo esperado?**

*Respuesta:* Sí, cualitativamente. Se espera que agentes que empiezan desordenados formen grupos alineados y compactos, y eso pasa: la polarización sube de ~0.1 a ~0.9 en 200 pasos y la distancia al vecino más cercano baja de ~1.8 a ~1.05 (sección 7). Dos matices:
- **La evolución no es monótona:** la polarización llega a ~0.79 en t = 50, baja a ~0.67 en t = 100 y sube otra vez. Probablemente son grupos que chocan con los bordes o entre sí y se reorganizan; conviene comprobarlo en la animación.
- **Hay bastante variabilidad entre semillas:** la polarización final va de ~0.69 a ~0.91 en 5 semillas (sección 5). Por eso hacen falta muchas réplicas.

**(b) Cómo verificar que el código hace lo previsto**

*Respuesta:*
- **Reproducibilidad:** la misma semilla da el mismo resultado (sección 6).
- **Invariantes:** en todo momento, cada velocidad mide 1 y cada posición está dentro del espacio.
- **Casos límite:**
  - Un solo agente no tiene vecinos y solo lo afecta la regla de bordes.
  - Con cohesión, separación y alineación en cero, no hay interacción y la polarización se queda baja.
  - Con alineación alta (`alignment_strength = 1`) se esperaría una polarización cercana a 1, pero la prueba da ~0.84, **menos** que con el valor base de 0.3 (~0.90). Es solo una semilla, pero sugiere que más alineación no siempre produce más orden: vale la pena revisarlo en el barrido de parámetros.
- **Revisión visual** con la animación.

La celda de abajo prueba los invariantes y los casos límite.

**(c) Limitaciones del modelo base**

*Respuesta:*
1. **Agentes idénticos:** mismos parámetros, sin roles ni diferencias individuales.
2. **Rapidez constante:** todos van a 1 unidad por paso; las reglas solo cambian la dirección.
3. **Percepción irreal:** 360°, perfecta, sin ruido y con todos los vecinos dentro de un radio fijo. Las aves reales interactúan con un número fijo de vecinos cercanos, unos 6 o 7 (Ballerini et al., 2008).
4. **Dinámica sin ruido:** el único azar está en las condiciones iniciales.
5. **Separación proporcional a la distancia**, en lugar de crecer al acercarse.
6. **Actualización secuencial** con una mezcla de velocidades viejas y nuevas (sección 5).
7. **Entorno vacío y cerrado:** el 64% del área está bajo influencia de los bordes, sin obstáculos, depredadores, recursos ni objetivos.
8. **Sin escala física:** las unidades de espacio y tiempo no corresponden a metros ni segundos.
9. **No registra datos:** solo se puede analizar visualmente.

In [ ]:
# Invariantes durante toda una corrida
m = BoidsModel(parameters2D)
m.sim_setup()
while m.running:
    m.sim_step()
    rapideces = np.linalg.norm([a.velocity for a in m.agents], axis=1)
    pos = np.array(list(m.space.positions.values()))
    assert np.allclose(rapideces, 1), f'Rapidez distinta de 1 en t={m.t}'
    assert (pos >= 0).all() and (pos <= m.p.size).all(), f'Agente fuera del espacio en t={m.t}'
print('Invariantes OK: rapidez = 1 y posiciones dentro del espacio en los', m.t, 'pasos')

# Casos límite
def pol_final(**cambios):
    m = BoidsModel(dict(parameters2D, **cambios))
    m.run(display=False)
    return polarizacion(m)

print(f"Un solo agente corre sin errores, polarización = {pol_final(population=1):.2f} (trivialmente 1)")
print(f"Sin interacción (cohesión = separación = alineación = 0): {pol_final(cohesion_strength=0, seperation_strength=0, alignment_strength=0):.3f}")
print(f"Alineación alta (alignment_strength = 1): {pol_final(alignment_strength=1):.3f}")

## 2. Parte II: mejoras candidatas y priorización

Las limitaciones del modelo base (sección 1, pregunta 9c) sugieren mejoras de cuatro tipos, según la parte del sistema que modifican:
- **Sensores:** qué percibe el ave.
- **Decisión:** cómo elige qué hacer con lo que percibe.
- **Estado interno y actuadores:** qué acumula el ave y qué tan capaz es de ejecutar su decisión.
- **Ambiente:** el espacio y lo que hay en él.

**Criterios de priorización**
- **Factibilidad** en el tiempo disponible.
- **Definición clara:** un mecanismo concreto con parámetros controlables.
- **Medible:** produce un indicador que lo distingue del modelo base.
- **Respaldo:** datos o literatura que justifiquen el mecanismo y sus valores.
- **Orden:** primero los cambios en el agente y después los del ambiente, para no tener que volver a modificar al agente al agregar elementos externos.

### 2.1 Mejoras candidatas

Cada candidata es un mecanismo mínimo. La factibilidad va de 1 (difícil) a 5 (sencilla).

| # | Mejora | Tipo | Criterios | Respaldo | Factibilidad | Decisión |
|---|---|---|---|---|---|---|
| 1 | Escala física (metros y segundos) | Ambiente | C2, C5, C6 | Rapidez de parvadas y tiempo de reacción del estornino | 5 | Etapa 0 |
| 2 | Giro máximo por paso | Actuador | C3 | Carga máxima en giros (Usherwood et al., 2011) | 5 | Etapa 0 |
| 3 | Espacio sin paredes (torus) | Ambiente | C2 | El cielo no tiene paredes; en el modelo base los bordes afectan al 64% del área | 4 | Etapa 0 |
| 4 | Radio de percepción ajustable | Sensor | C3 | Parámetro de control para los experimentos | 5 | Mejora 1 |
| 5 | Vecinos topológicos (los *k* más cercanos) | Sensor | C3, C6 | Los estorninos siguen a 6 o 7 vecinos (Ballerini et al., 2008a) | 4 | Mejora 1 |
| 6 | Campo visual con punto ciego trasero | Sensor | C3 | Punto ciego trasero amplio en el estornino (Tisdale y Fernández-Juricic, 2009) | 4 | Mejora 1 |
| 7 | Ruido en la percepción | Sensor | C4 | Supuesto | 4 | Futuro |
| 8 | Líderes informados con una dirección preferida | Decisión | C1, C3 | Una minoría informada basta para guiar al grupo (Couzin et al., 2005) | 4 | Mejora 2 |
| 9 | Proporción de cada rol controlable | Decisión | C1, C8 | Permite barrer la fracción de líderes y de juveniles | 5 | Mejora 2 |
| 10 | Parámetros propios de cada rol | Decisión | C1 | Cualquier regla puede tener un valor distinto por rol | 5 | Mejora 2 |
| 11 | Juveniles con mayor gasto de vuelo | Estado | C1 | Los juveniles gastan ~14% más en vuelo (Rotics et al., 2016) | 4 | Mejora 2 y 3 |
| 12 | Retraso de reacción en juveniles | Decisión | C1 | No se encontró evidencia | — | Descartada |
| 13 | Gasto de energía según la rapidez | Estado | C3, C6 | Potencia de vuelo del estornino (Ward et al., 2001) | 4 | Mejora 3 |
| 14 | Gasto de energía en los giros | Estado | C3 | Duplicar la carga en un giro cuadruplica el esfuerzo (Usherwood et al., 2011); girar es costoso (Wilson et al., 2013) | 4 | Mejora 3 |
| 15 | Gasto creciente por giros consecutivos, con tope | Estado | C3 | Supuesto razonado | 3 | Mejora 3 |
| 16 | Recuperación de energía distinta en cada ave, con umbral | Estado | C1, C4 | La tasa metabólica varía de forma consistente entre individuos (Versteegh et al., 2008) | 4 | Mejora 3 |
| 17 | Rapidez máxima que depende de la energía | Actuador | C3 | Rango de rapidez observado (Ballerini et al., 2008b) | 4 | Mejora 3 |
| 18 | Masa corporal | Estado | C1 | Su efecto principal ya se representa con el gasto de energía | — | Descartada |
| 19 | Edad | Estado | C1 | Sin un mecanismo propio; su efecto (experiencia) ya se representa con los roles | — | Descartada |
| 20 | Actualización sincrónica de velocidades | Decisión | C5 | El orden de activación tiene un efecto agregado pequeño (sección 1, pregunta 5) | 5 | Futuro |
| 21 | Separación que crece al acercarse | Decisión | C3 | Limitación del modelo base, compensada con la calibración (sección 4.1) | 4 | Futuro |
| 22 | Viento variable en el espacio y el tiempo | Ambiente | C2, C4 | Un viento uniforme no cambia la estructura de la parvada; uno variable sí | 3 | Futuro |
| 23 | Obstáculos | Ambiente | C2 | — | 3 | Futuro |
| 24 | Depredador | Ambiente | C1, C2 | Requiere un segundo tipo de agente | 2 | Futuro |
| 25 | Espacio en 3D | Ambiente | C2 | Los datos de referencia son 3D; aumenta el costo de cómputo | 3 | Futuro |

### 2.2 Mejoras elegidas

**Etapa 0, modelo de referencia (candidatas 1 a 3).** No es una mejora de comportamiento, sino el requisito para comparar con datos reales: da escala física al modelo y corrige lo que no es físicamente posible.

1. **Percepción realista (candidatas 4 a 6).** Cada ave responde solo a sus *k* vecinos más cercanos, dentro de un radio y fuera de su punto ciego.
2. **Roles (candidatas 8 a 11).** Las aves se dividen en líderes informados, adultos y juveniles, en proporciones controlables y con parámetros propios por rol. Los líderes tienen una dirección preferida, que tratan de seguir sin dejar de responder a sus vecinos.
3. **Energía (candidatas 13 a 17 y 11).** Volar rápido y girar gastan energía, y los giros seguidos gastan cada vez más, hasta un tope. Cada ave recupera energía a su propio ritmo cuando su gasto queda por debajo de ese ritmo. Con poca energía, baja su rapidez máxima.

**Cómo se complementan.** Las tres responden una misma pregunta: **¿cómo se mantiene unida una parvada cuyos miembros difieren en lo que perciben, lo que saben y lo que pueden sostener?** La percepción define a quién responde cada ave; los roles introducen información desigual (líderes) y capacidad desigual (juveniles); y la energía convierte esa diferencia de capacidad en aves rezagadas. Con eso se pueden estudiar dos preguntas concretas:
- ¿Cuántos líderes bastan para guiar a la parvada?
- Cuando algunas aves se cansan, ¿el grupo se adapta a ellas o las deja atrás?

## 3. Parte III: modelo extendido

Las mejoras se construyen por etapas. Cada etapa es una clase que hereda de la anterior y agrega un solo mecanismo. Con sus parámetros nuevos en el valor por defecto, cada etapa se comporta igual que la anterior; así se puede medir el efecto de cada mejora por separado (sección 6).

| Etapa | Sección | Qué agrega |
|---|---|---|
| 0 | 3.0 | Escala realista y espacio sin paredes (modelo de referencia) |
| 1 | 3.1 | Percepción |
| 2 | 3.2 | Roles |
| 3 | 3.3 | Energía |

En las tablas de parámetros, la columna **Fuente** indica si el valor es un **dato** medido (con su cita), un valor **derivado** de datos, un valor **calibrado** (sección 4) o un **supuesto** razonado.

### 3.0 Modelo de referencia: escala realista

El modelo base no tiene unidades: el espacio mide 50 × 50 "unidades" y un paso es un instante sin duración. Para poder compararlo con aves reales, esta etapa fija una escala física y corrige los aspectos del modelo base que no son físicamente posibles. No agrega comportamientos nuevos: es la **referencia** contra la que se comparan las mejoras.

**Escala.** 1 unidad = 1 m y 1 paso = 0,1 s.
- Con esta escala, la rapidez del modelo base (1 unidad por paso) equivale a **10 m/s**. Coincide con la rapidez de parvadas reales de estornino, que en 10 parvadas medidas en 3D va de 6,9 a 15,2 m/s, con media de 10,6 m/s (Ballerini et al., 2008b).
- 0,1 s es del orden del tiempo de reacción del estornino, 76 ms (Pomeroy y Heppner, 1977). Cada paso corresponde aproximadamente a una decisión del ave.

**Cambios respecto al modelo base**
1. **Rumbo y rapidez separados.** La velocidad se descompone en un rumbo unitario (`heading`) y una rapidez (`speed`). En esta etapa la rapidez sigue siendo constante; la Etapa 3 la hace variable.
2. **Giro máximo por paso.** En el modelo base un agente puede girar cualquier ángulo en un paso. Un ave gira inclinándose, y la fuerza que soporta en el giro es limitada: las palomas alcanzan hasta 2 g en giros con aleteo (Usherwood et al., 2011). Con factor de carga *n*, la aceleración lateral máxima es *a* = *g*·√(*n*² − 1), y el giro máximo por paso es Δθ = *a*·Δ*t* / *v*. Con *n* = 2, el giro máximo es de ≈ 9,7° por paso a 10 m/s. No se encontró un valor publicado para estornino, así que el de paloma se usa como aproximación.
3. **Espacio sin paredes.** El cielo no tiene paredes, y en el modelo base la regla de bordes afecta al 64% del área (sección 1, pregunta 3). Aquí el espacio es un **torus**: lo que sale por un lado entra por el opuesto. Las reglas usan el desplazamiento más corto entre dos aves, que puede cruzar el borde, y la regla de bordes desaparece.
4. **Parvada ya formada al inicio.** En el modelo base las aves empiezan dispersas por todo el espacio y con rumbos aleatorios, y la simulación muestra cómo se *forma* una parvada. Aquí empiezan como una parvada ya formada, porque las parvadas reales que sirven de referencia ya lo estaban (Ballerini et al., 2008b; Cavagna et al., 2010) y porque la pregunta de este trabajo es cómo se *mantiene* la parvada. Además, con la percepción realista de la Etapa 1 las aves no logran formar una sola parvada a partir del desorden (sección 3.1).
   - **Posición:** un parche cuadrado en el centro, con distancia media al vecino de 1 m. Así la densidad inicial no depende del tamaño del espacio.
   - **Rumbo:** una dirección común, elegida al azar, más una desviación normal con desviación estándar σ. En parvadas de estornino, la polarización media es Φ = 0,96 ± 0,03 (Cavagna et al., 2010). Si los rumbos siguen una distribución normal envuelta, Φ = e^(−σ²/2), así que σ = √(−2 ln 0,96) ≈ **16,4°**. Con ese valor, la polarización inicial del modelo es 0,96.
   - Con `init_heading_sd = None` se recuperan los rumbos aleatorios del modelo base.

**Tamaño del torus.** Debe ser lo bastante grande para que la parvada no se alcance a sí misma por el otro lado: la extensión de la parvada más dos radios de percepción debe ser menor que el lado. Con 200 aves y ~1 m entre vecinas, una parvada compacta ocupa unos 24 × 8 m (área ≈ *N*·*d*², con relación de aspecto ~3, dentro del rango 1,9–3,6 medido por Ballerini et al., 2008b). Se usa un lado de 200 m; la sección 4.1 verifica que la extensión observada cumple la condición.

#### Utilidades geométricas

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

G = 9.81  # Aceleración de la gravedad (m/s²)


def desplazamiento_toroidal(origen, destino, size):
    """ Vector más corto de `origen` a `destino` en un torus cuadrado de lado `size`.
    Funciona con puntos sueltos o con arreglos de puntos (broadcasting de numpy). """
    d = np.asarray(destino) - np.asarray(origen)
    return d - size * np.round(d / size)


def rotar_hacia(heading, deseada, max_angulo):
    """ Gira el vector unitario `heading` hacia la dirección `deseada` (2D),
    como máximo `max_angulo` radianes. Devuelve el nuevo rumbo y el ángulo girado. """
    angulo = np.arctan2(heading[0] * deseada[1] - heading[1] * deseada[0],
                        heading @ deseada)
    giro = np.clip(angulo, -max_angulo, max_angulo)
    c, s = np.cos(giro), np.sin(giro)
    nuevo = np.array([c * heading[0] - s * heading[1],
                      s * heading[0] + c * heading[1]])
    return nuevo, giro

#### Agente

In [ ]:
class BoidReferencia(ap.Agent):
    """ Boid con escala física: 1 unidad = 1 m y 1 paso = dt segundos.

    Diferencias con el modelo base:
    - Dirección (`heading`) y rapidez (`speed`) separadas; `velocity` = heading × speed.
    - Giro máximo por paso, limitado por el factor de carga máximo del vuelo.
    - Vecindad en un torus: las reglas usan el desplazamiento más corto.
    - Sin regla de bordes, porque el torus no tiene paredes. """

    def setup(self):
        sd = self.p.get('init_heading_sd', None)
        if sd is None:
            # Parvada sin formar: rumbo aleatorio, como en el modelo base
            self.heading = normalize(self.model.nprandom.random(self.p.ndim) - 0.5)
        else:
            # Parvada ya formada: rumbo común con dispersión normal de `sd` grados
            angulo = self.model.direccion_inicial + self.model.nprandom.normal(0, np.radians(sd))
            self.heading = np.array([np.cos(angulo), np.sin(angulo)])
        self.speed = self.p.speed_pref
        self.velocity = self.heading * self.speed
        self.turn = 0.0  # Ángulo girado en el último paso (rad)

    def setup_pos(self, space):
        self.space = space
        self.pos = space.positions[self]

    def max_turn(self):
        """ Giro máximo por paso (rad) a la rapidez actual.
        Aceleración lateral máxima a = g·√(n² − 1); giro Δθ = a·dt / v,
        con v = speed / dt la rapidez en m/s. """
        a_max = G * np.sqrt(self.p.max_load_factor ** 2 - 1)
        return a_max * self.p.dt ** 2 / self.speed

    def vecinos(self):
        """ Vecinos que percibe el ave: desplazamientos hacia ellos y sus velocidades.
        En la referencia, todas las aves a menos de outer_radius. """
        nbs = self.space.neighbors(self, distance=self.p.outer_radius)
        if len(nbs) == 0:
            return np.zeros((0, self.p.ndim)), np.zeros((0, self.p.ndim))
        disp = desplazamiento_toroidal(self.pos, np.array(nbs.pos), self.p.size)
        return disp, np.array(nbs.velocity)

    def social_force(self):
        """ Suma de las reglas de cohesión, separación y alineación. """
        disp, vel = self.vecinos()
        force = np.zeros(self.p.ndim)
        if len(disp) == 0:
            return force
        dist = np.linalg.norm(disp, axis=1)

        # Regla 1 - Cohesión: hacia el centro de los vecinos
        force += disp.mean(axis=0) * self.p.cohesion_strength
        # Regla 2 - Separación: alejarse de los vecinos dentro de inner_radius
        force -= disp[dist < self.p.inner_radius].sum(axis=0) * self.p.separation_strength
        # Regla 3 - Alineación: igualar la velocidad promedio de los vecinos
        force += (vel.mean(axis=0) - self.velocity) * self.p.alignment_strength
        return force

    def update_velocity(self):
        deseada = self.velocity + self.social_force()
        self.heading, self.turn = rotar_hacia(self.heading, deseada, self.max_turn())
        self.velocity = self.heading * self.speed

    def update_position(self):
        self.space.move_to(self, np.mod(self.pos + self.velocity, self.p.size))

#### Modelo

In [ ]:
class ModeloReferencia(ap.Model):
    """ Modelo de boids con escala realista en un torus. Es la referencia
    contra la que se comparan las mejoras. """

    agent_class = BoidReferencia

    def setup(self):
        assert self.p.ndim == 2, 'El modelo de referencia es bidimensional'
        self.direccion_inicial = self.nprandom.uniform(0, 2 * np.pi)  # Rumbo común de la parvada (rad)
        self.space = ap.Space(self, shape=[self.p.size] * 2, torus=True)
        self.agents = ap.AgentList(self, self.p.population, self.agent_class)
        self.space.add_agents(self.agents, positions=self.posiciones_iniciales())
        self.agents.setup_pos(self.space)

    def posiciones_iniciales(self):
        """ Parche cuadrado en el centro del espacio. Para puntos uniformes con
        densidad ρ, la distancia media al vecino más cercano es ≈ 0.5/√ρ; el lado
        del parche se elige para que esa distancia sea `init_nnd`. """
        lado = 2 * self.p.init_nnd * np.sqrt(self.p.population)
        centro = self.p.size / 2
        return [centro + (self.nprandom.random(2) - 0.5) * lado
                for _ in range(self.p.population)]

    def step(self):
        self.agents.update_velocity()
        self.agents.update_position()

#### Métricas de la parvada

Se calculan sobre las distancias del torus:
- **Polarización:** magnitud del rumbo promedio (0 = desorden, 1 = todos en la misma dirección). En parvadas reales de estornino es 0,96 ± 0,03 (Cavagna et al., 2010).
- **Distancia al vecino:** promedio de la distancia de cada ave a su vecina más cercana.
- **Fracción en colisión:** aves con una vecina a menos de 0,38 m, la distancia mínima observada entre estorninos (Ballerini et al., 2008b).
- **Grupos:** dos aves pertenecen al mismo grupo si están conectadas por una cadena de vecinas a menos de `outer_radius`. Se reporta el número de grupos y la fracción de aves en el mayor.
- **Extensión:** mayor distancia entre dos aves del grupo mayor.

In [ ]:
def distancias_toroidales(m):
    """ Matriz de distancias entre todos los agentes, por el camino más corto en el torus. """
    pos = np.array(list(m.space.positions.values()))
    return np.linalg.norm(desplazamiento_toroidal(pos[:, None], pos[None], m.p.size), axis=-1)


def metricas_parvada(m, d=None):
    """ Indicadores del estado de la parvada en el paso actual.
    `d` es opcional: la matriz de distancias ya calculada, con infinito en la diagonal. """
    if d is None:
        d = distancias_toroidales(m)
        np.fill_diagonal(d, np.inf)
    nnd = d.min(axis=1)
    n_grupos, etiquetas = connected_components(csr_matrix(d < m.p.outer_radius), directed=False)
    tamanos = np.bincount(etiquetas)
    mayor = etiquetas == tamanos.argmax()
    d_mayor = d[np.ix_(mayor, mayor)]
    extension = d_mayor[np.isfinite(d_mayor)].max() if mayor.sum() > 1 else 0.0
    return {
        'polarizacion': np.linalg.norm(np.mean([a.heading for a in m.agents], axis=0)),
        'dist_vecino': nnd.mean(),
        'frac_colision': (nnd < m.p.collision_distance).mean(),
        'n_grupos': n_grupos,
        'frac_grupo_mayor': tamanos.max() / len(etiquetas),
        'extension': extension,
    }


class ModeloReferenciaConDatos(ModeloReferencia):
    """ Modelo de referencia que registra las métricas en cada paso. """

    def update(self):
        for k, v in metricas_parvada(self).items():
            self.record(k, v)

#### Parámetros

| Parámetro | Valor | Unidad | Fuente |
|---|---|---|---|
| `dt` | 0,1 | s | Dato: tiempo de reacción de 76 ms (Pomeroy y Heppner, 1977) |
| `speed_pref` | 1,0 | m/paso (10 m/s) | Dato: rapidez de parvadas, 6,9–15,2 m/s (Ballerini et al., 2008b) |
| `max_load_factor` | 2 | *g* | Dato en paloma (Usherwood et al., 2011), usado como aproximación |
| `size` | 200 | m | Derivado; verificado en la sección 4.1 |
| `population` | 200 | aves | Supuesto, por costo de cómputo; las parvadas medidas tienen 448–2631 aves |
| `init_nnd` | 1,0 | m | Dato: distancia al vecino observada (Ballerini et al., 2008b) |
| `init_heading_sd` | 16,4 | grados | Derivado de la polarización observada, 0,96 (Cavagna et al., 2010) |
| `collision_distance` | 0,38 | m | Dato: distancia mínima entre aves (Ballerini et al., 2008b) |
| `inner_radius` | 1,0 | m | Calibrado (sección 4.1) |
| `separation_strength` | 0,4 | — | Calibrado (sección 4.1) |
| `outer_radius` | 10 | m | Modelo base |
| `cohesion_strength` | 0,005 | — | Modelo base |
| `alignment_strength` | 0,3 | — | Modelo base |

In [ ]:
parametros_ref = {
    # Simulación
    'seed': 123,
    'steps': 1000,             # 1000 pasos = 100 s de vuelo
    'ndim': 2,
    'population': 200,
    # Escala
    'dt': 0.1,                 # s por paso
    'size': 200,               # m, lado del torus
    'init_nnd': 1.0,           # m, distancia al vecino en la posición inicial
    'init_heading_sd': 16.4,   # grados, dispersión inicial de rumbos (None = rumbos aleatorios)
    'collision_distance': 0.38,  # m, distancia mínima observada entre aves
    # Vuelo
    'speed_pref': 1.0,         # m/paso (= 10 m/s)
    'max_load_factor': 2.0,    # g
    # Reglas
    'inner_radius': 1.0,         # m, calibrado en la sección 4.1
    'outer_radius': 10,        # m
    'cohesion_strength': 0.005,
    'separation_strength': 0.4,  # calibrado en la sección 4.1
    'alignment_strength': 0.3,
}

#### Simulación de referencia

Una corrida de 1000 pasos (100 s de vuelo).

In [ ]:
ref = ModeloReferenciaConDatos(parametros_ref)
salida_ref = ref.run(display=False)
serie_ref = salida_ref.variables.ModeloReferenciaConDatos

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
serie_ref['polarizacion'].plot(ax=axs[0], title='Polarización')
axs[0].axhspan(0.93, 0.99, color='green', alpha=0.15, label='Rango observado')
axs[0].legend()
serie_ref['dist_vecino'].plot(ax=axs[1], title='Distancia al vecino (m)')
axs[1].axhspan(0.68, 1.51, color='green', alpha=0.15, label='Rango observado')
axs[1].legend()
pos = np.array(list(ref.space.positions.values()))
rumbos = np.array([a.heading for a in ref.agents])
axs[2].quiver(pos[:, 0], pos[:, 1], rumbos[:, 0], rumbos[:, 1], angles='xy', scale=40, width=0.003)
axs[2].set(xlim=(0, ref.p.size), ylim=(0, ref.p.size), aspect='equal', title=f't = {ref.t} (posición y rumbo)')
for ax in axs[:2]:
    ax.set_xlabel('Paso t')
plt.tight_layout()
plt.show()
serie_ref.loc[[0, 100, 250, 500, 1000]].round(3)

### 3.1 Mejora 1: percepción realista

En la referencia, cada ave percibe a **todas** las aves a menos de 10 m, en todas direcciones y sin límite de número. Esta mejora cambia los **sensores** del ave con tres mecanismos:

1. **Alcance de visión** (`vision_radius`): distancia máxima a la que el ave ve a otra.
2. **Punto ciego trasero** (`vision_angle`): el ave solo ve dentro de un ángulo centrado en su rumbo. Las aves con ojos laterales, como el estornino, ven casi todo a su alrededor excepto una zona detrás de la cabeza, que en el estornino es amplia en comparación con otras aves (Tisdale y Fernández-Juricic, 2009).
3. **Interacción topológica** (`n_neighbors`): el ave responde solo a sus *k* vecinos visibles más cercanos, sin importar a qué distancia estén. En parvadas reales de estornino, cada ave interactúa con 6 o 7 vecinos, no con todas las aves dentro de una distancia fija (Ballerini et al., 2008a).

Las tres reglas (cohesión, separación y alineación) usan el mismo conjunto de vecinos percibidos.

| Parámetro | Valor por defecto (= referencia) | Valor realista | Fuente del valor realista |
|---|---|---|---|
| `vision_radius` | 10 m | 50 m | Supuesto: mayor que la parvada, para que la interacción dependa del número de vecinos y no de la distancia |
| `vision_angle` | 360° | 300° | Supuesto: punto ciego trasero de 60°. No se encontró un valor publicado accesible; se analiza la sensibilidad (sección 6) |
| `n_neighbors` | todos (`None`) | 7 | Dato: 6 a 7 vecinos (Ballerini et al., 2008a) |

El campo visual real es tridimensional; en este modelo bidimensional se representa en el plano horizontal.

**Implementación.** El ave consulta el árbol k-d del espacio, que ya considera el torus, para obtener sus vecinos más cercanos dentro de `vision_radius`. Descarta los que caen en su punto ciego y se queda con los *k* más cercanos. Si el punto ciego deja menos de *k* candidatos, amplía la búsqueda. Con los valores por defecto usa exactamente el método de la referencia.

#### Agente

In [ ]:
class BoidPercepcion(BoidReferencia):
    """ Boid con percepción realista:
    - ve solo dentro de su campo visual (`vision_angle`, centrado en su rumbo),
      es decir, no percibe a las aves en su punto ciego trasero;
    - ve hasta `vision_radius` metros;
    - responde solo a sus `n_neighbors` vecinos visibles más cercanos
      (interacción topológica). Con `n_neighbors = None` responde a todos.

    Con vision_angle = 360, n_neighbors = None y vision_radius = outer_radius
    se comporta exactamente igual que BoidReferencia. """

    def vecinos(self):
        p = self.p
        radio = p.get('vision_radius', p.outer_radius)
        angulo = p.get('vision_angle', 360)
        k = p.get('n_neighbors', None)
        if k is None and angulo >= 360 and radio == p.outer_radius:
            return super().vecinos()

        tree = self.space.kdtree
        agentes = self.space._sorted_agents  # Agentes en el orden de los puntos del árbol k-d
        n = len(agentes)
        candidatos = n if k is None else min(n, 3 * k + 1)
        while True:
            if k is None:
                idx = np.array(tree.query_ball_point(self.pos, radio), dtype=int)
            else:
                dist, idx = tree.query(self.pos, k=candidatos, distance_upper_bound=radio)
                idx = np.atleast_1d(idx)[np.isfinite(np.atleast_1d(dist))]
            idx = idx[[agentes[i] is not self for i in idx]]

            disp = desplazamiento_toroidal(self.pos, tree.data[idx], p.size)
            dist = np.linalg.norm(disp, axis=1)
            if angulo < 360:
                # Visible si el ángulo entre el rumbo y la dirección al vecino es ≤ angulo/2
                cos_vecino = disp @ self.heading / np.maximum(dist, 1e-12)
                visible = cos_vecino >= np.cos(np.radians(angulo / 2))
                idx, disp, dist = idx[visible], disp[visible], dist[visible]

            # Si la búsqueda se truncó y quedaron menos de k visibles, se amplía
            if k is None or len(idx) >= k or candidatos >= n:
                break
            candidatos = min(n, 2 * candidatos)

        if k is not None:
            orden = np.argsort(dist)[:k]
            idx, disp = idx[orden], disp[orden]
        if len(idx) == 0:
            return np.zeros((0, p.ndim)), np.zeros((0, p.ndim))
        return disp, np.array([agentes[i].velocity for i in idx])

#### Modelo y parámetros

In [ ]:
class ModeloPercepcion(ModeloReferencia):
    """ Modelo de la Etapa 1: referencia + percepción realista. """
    agent_class = BoidPercepcion


class ModeloPercepcionConDatos(ModeloReferenciaConDatos):
    """ Etapa 1 con el registro de métricas de la sección 3.0. """
    agent_class = BoidPercepcion

In [ ]:
parametros_percepcion = dict(
    parametros_ref,
    vision_radius=50,   # m
    vision_angle=300,   # grados; punto ciego trasero de 60°
    n_neighbors=7,      # vecinos con los que interactúa cada ave
)

#### Hallazgo: formar una parvada no es lo mismo que mantenerla

Se compara la percepción de la referencia, la interacción con 7 vecinos sin punto ciego y la percepción realista completa, en dos condiciones iniciales: aves con rumbos aleatorios (parvada desordenada, como en el modelo base) y la parvada ya formada de la sección 3.0. Son 20 réplicas por configuración, con métricas promediadas entre los pasos 500 y 1000.

In [ ]:
percepciones = {
    'Referencia': {},
    '7 vecinos, sin punto ciego': dict(n_neighbors=7, vision_radius=50),
    'Percepción realista': dict(n_neighbors=7, vision_radius=50, vision_angle=300),
}
inicios = {'Desordenada': None, 'Formada': parametros_ref['init_heading_sd']}

filas = []
for percepcion, cambios in percepciones.items():
    for inicio, sd in inicios.items():
        muestra = ap.Sample(dict(parametros_ref, init_heading_sd=sd, **cambios), n=1)
        exp = ap.Experiment(ModeloPercepcionConDatos, muestra, iterations=20, record=True, randomize=True)
        v = exp.run(n_jobs=-1, verbose=0).variables.ModeloPercepcionConDatos.reset_index()
        por_corrida = v[v.t >= 500].groupby('iteration')[['polarizacion', 'n_grupos', 'frac_grupo_mayor', 'dist_vecino']].mean()
        filas.append({'percepción': percepcion, 'inicio': inicio,
                      **{f'{k} (media)': por_corrida[k].mean() for k in por_corrida},
                      **{f'{k} (desv.)': por_corrida[k].std() for k in por_corrida}})
formacion = pd.DataFrame(filas).set_index(['percepción', 'inicio'])

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, kpi, titulo in [(axs[0], 'frac_grupo_mayor', 'Fracción de aves en el grupo mayor'),
                        (axs[1], 'polarizacion', 'Polarización')]:
    tabla = formacion[f'{kpi} (media)'].unstack('inicio').loc[list(percepciones)]
    error = formacion[f'{kpi} (desv.)'].unstack('inicio').loc[list(percepciones)]
    tabla.plot.bar(yerr=error, ax=ax, rot=0, capsize=3, title=titulo)
    ax.set_xlabel('')
axs[1].axhspan(0.93, 0.99, color='green', alpha=0.15, label='Rango observado')
axs[1].legend(title='Inicio')
plt.tight_layout()
plt.show()
formacion.round(3)

**Resultados**

- **Con la parvada desordenada, la percepción realista no forma una sola parvada en 100 s.** Termina en ~5 grupos y el mayor reúne ~41% de las aves. Con 7 vecinos y sin punto ciego es peor: ~9 grupos, con el mayor en ~27%. La referencia, en la que cada ave ve a todas las de un radio de 10 m (unas 50 en el parche inicial), termina en 2 o 3 grupos.
- **Con la parvada ya formada, la percepción realista la mantiene:** ~94% de las aves sigue en el grupo mayor, con polarización de 0,995.

**Interpretación.** Con solo 7 vecinos, la dirección se transmite de vecino en vecino. Desde el desorden, grupos pequeños llegan a un consenso local en direcciones distintas y se separan antes de coordinarse con el resto. Con más vecinos por ave, como en la referencia, el consenso es más global. Esto no contradice a Ballerini et al. (2008a), que observaron parvadas ya formadas; indica que la interacción topológica sirve para **mantener** una parvada, no necesariamente para **formarla** desde aves dispersas. Por esta razón, la referencia y todas las etapas empiezan con la parvada ya formada (sección 3.0).

**El punto ciego ayuda a mantener la parvada unida:** con la parvada formada, la fracción en el grupo mayor sube de ~79% sin punto ciego a ~94% con él. Una posible explicación, que no se verifica aquí, es que al ignorar a las aves de atrás cada ave sigue sobre todo a las que van delante y a los lados, y la dirección se transmite de forma más ordenada hacia atrás. El efecto de cada parámetro por separado se analiza en la sección 6.

### 3.2 Mejora 2: roles

En la referencia todas las aves son idénticas. Esta mejora introduce dos tipos de diferencia entre individuos: de **información** (algunas aves saben hacia dónde ir) y de **capacidad** (algunas aves se cansan más). Para eso, cada ave tiene uno de tres roles:

| Rol | Proporción por defecto | Comportamiento | Respaldo |
|---|---|---|---|
| **Líder informado** | 10% | Reglas sociales + atracción hacia una dirección preferida | Una minoría informada basta para guiar a un grupo, y cuanto mayor el grupo, menor la proporción necesaria (Couzin et al., 2005). La proporción se elige con el experimento de abajo |
| **Adulto** | 45% (el resto) | Reglas sociales | — |
| **Juvenil** | 45% | Reglas sociales; en la Etapa 3, gasta 14% más energía al volar | Los juveniles de cigüeña gastan ~14% más en vuelo que los adultos y vuelan detrás de ellos (Rotics et al., 2016) |

- **Proporción de juveniles (derivada).** En estorninos, la supervivencia anual es de ~0,518 en el primer año (BTO, s. f.) y de ~0,45 en adultos (Bateson y Asher, 2010). En una población estable, los adultos que mueren cada año se reponen con juveniles: por cada adulto hay (1 − 0,45) / 0,518 ≈ 1,06 juveniles después de la reproducción, es decir, ~51% de juveniles. A mitad del invierno, tras la mortalidad juvenil, la fracción baja a ~43%. Se usa 45%.
- **Los líderes son adultos.** Los juveniles migran detrás de los adultos (Rotics et al., 2016), así que la información se asigna solo a adultos.
- **En esta etapa, juveniles y adultos se comportan igual.** Su diferencia, el gasto de energía, aparece en la Etapa 3.

**Proporciones exactas.** `frac_lider` y `frac_juvenil` fijan cuántas aves tiene cada rol (redondeando de modo que la suma sea la población) y el resto son adultos. El orden se baraja con el generador del modelo, así que la asignación es reproducible con la semilla.

**Parámetros por rol.** Cada ave tiene su propia copia de los parámetros. Un parámetro con sufijo `__<rol>` reemplaza al parámetro base solo para ese rol; por ejemplo, `alignment_strength__juvenil = 0.1` cambia la alineación solo de los juveniles. Así cualquier regla puede tener un valor distinto por rol sin cambiar su código, y ese valor se puede barrer en un experimento (`ap.Experiment` no admite diccionarios como parámetros).

**Regla de los líderes.** Siguiendo a Couzin et al. (2005), el líder combina la dirección que le dictan sus vecinos con una dirección preferida *g*, común a todos los líderes y fija durante la corrida: a su velocidad deseada se suma `goal_weight` × rapidez × *g*. La dirección se elige al azar en cada corrida.

Se descartó una versión con un **punto destino** que se renovaba al alcanzarlo. En pruebas preliminares, con peso bajo la parvada no seguía a los líderes, y con peso alto los líderes alcanzaban un destino cada ~10 s y el giro brusco hacia cada destino nuevo rompía la parvada. Además, en el torus un destino no puede estar a más de ~100 m. Una dirección fija evita ambos problemas y es la formulación original de Couzin et al. (2005).

| Parámetro | Valor | Fuente |
|---|---|---|
| `frac_lider` | 0,10 | Elegido con el experimento de abajo |
| `frac_juvenil` | 0,45 | Derivado de la supervivencia anual (BTO, s. f.; Bateson y Asher, 2010) |
| `goal_weight` | 0,05 | Supuesto; su efecto se analiza en la sección 6 |

#### Utilidades

In [ ]:
ROLES = ('lider', 'adulto', 'juvenil')


def conteo_por_rol(fracciones, n):
    """ Número exacto de aves por rol. Redondea hacia abajo y reparte el resto
    entre los roles con mayor parte fraccionaria, para que la suma sea `n`. """
    ideal = {r: fracciones[r] * n for r in fracciones}
    conteo = {r: int(np.floor(x)) for r, x in ideal.items()}
    faltan = n - sum(conteo.values())
    for r in sorted(ideal, key=lambda r: ideal[r] - conteo[r], reverse=True)[:faltan]:
        conteo[r] += 1
    return conteo

#### Agente

In [ ]:
class BoidRoles(BoidPercepcion):
    """ Boid con un rol: 'lider', 'adulto' o 'juvenil'.

    - Cada ave tiene su propia copia de parámetros: los del modelo, sobrescritos
      por los de su rol. Un parámetro con sufijo `__<rol>` (por ejemplo
      `alignment_strength__juvenil`) reemplaza al parámetro base solo para ese
      rol. Así cualquier regla puede tener un valor distinto por rol sin cambiar
      su código, y ese valor se puede barrer en un experimento.
    - Los líderes son adultos informados: además de las reglas sociales, giran
      hacia la dirección preferida del modelo con peso `goal_weight`
      (Couzin et al., 2005). """

    def setup(self):
        self.role = next(self.model.roles_por_asignar)
        sufijo = f'__{self.role}'
        propios = {k[:-len(sufijo)]: v for k, v in self.model.p.items() if k.endswith(sufijo)}
        self.p = ap.AttrDict({**self.model.p, **propios})
        super().setup()

    def goal_force(self):
        """ Atracción hacia la dirección preferida (solo líderes), con la magnitud de la velocidad. """
        if self.role != 'lider' or self.p.goal_weight == 0:
            return np.zeros(self.p.ndim)
        return self.p.goal_weight * self.speed * self.model.goal_dir

    def social_force(self):
        return super().social_force() + self.goal_force()

#### Modelo y parámetros

In [ ]:
class ModeloRoles(ModeloPercepcion):
    """ Modelo de la Etapa 2: percepción realista + roles.

    Las proporciones son exactas: `frac_lider` y `frac_juvenil` fijan cuántas aves
    tiene cada rol y el resto son adultos; el orden se baraja con el generador del
    modelo. Si hay líderes, el modelo elige al azar una dirección preferida
    (`goal_dir`), común a todos ellos y fija durante la corrida. """

    agent_class = BoidRoles

    def setup(self):
        fracciones = {'lider': self.p.get('frac_lider', 0.0),
                      'juvenil': self.p.get('frac_juvenil', 0.0)}
        fracciones['adulto'] = 1 - sum(fracciones.values())
        assert fracciones['adulto'] >= 0, 'frac_lider + frac_juvenil no puede superar 1'
        self.conteo_roles = conteo_por_rol(fracciones, self.p.population)
        roles = [r for r in ROLES for _ in range(self.conteo_roles[r])]
        self.random.shuffle(roles)
        self.roles_por_asignar = iter(roles)
        super().setup()
        if self.conteo_roles['lider'] > 0:
            angulo = self.random.uniform(0, 2 * np.pi)
            self.goal_dir = np.array([np.cos(angulo), np.sin(angulo)])


class ModeloRolesConDatos(ModeloRoles):
    """ Etapa 2 con el registro de la sección 3.0, más la precisión de rumbo. """

    def update(self):
        ModeloReferenciaConDatos.update(self)
        if hasattr(self, 'goal_dir'):
            self.record('precision_rumbo', np.mean([a.heading @ self.goal_dir for a in self.agents]))

In [ ]:
parametros_roles = dict(
    parametros_percepcion,
    frac_lider=0.10,      # líderes informados (adultos)
    frac_juvenil=0.45,    # juveniles; el resto son adultos
    goal_weight=0.05,     # peso de la dirección preferida (solo líderes)
)

#### Efecto de los líderes según la percepción

Se varía la fracción de líderes (0%, 5%, 10% y 20%) con la percepción de la referencia y con la percepción realista. La **precisión de rumbo** es el promedio del coseno entre el rumbo de cada ave y la dirección preferida: 1 si todas siguen esa dirección, 0 si no hay relación y −1 si van en sentido contrario. Son 20 réplicas por configuración, con métricas promediadas entre los pasos 500 y 1000.

In [ ]:
percepciones = {
    'Referencia': dict(n_neighbors=None, vision_angle=360, vision_radius=10),
    'Percepción realista': {},
}
fracciones_lider = ap.Values(0.0, 0.05, 0.1, 0.2)

tablas = []
for percepcion, cambios in percepciones.items():
    muestra = ap.Sample(dict(parametros_roles, frac_lider=fracciones_lider, **cambios))
    exp = ap.Experiment(ModeloRolesConDatos, muestra, iterations=20, record=True, randomize=True)
    v = exp.run(n_jobs=-1, verbose=0).arrange_variables()
    por_corrida = (v[v.t >= 500].groupby(['frac_lider', 'iteration'])
                   [['precision_rumbo', 'frac_grupo_mayor', 'n_grupos', 'polarizacion']].mean())
    resumen = por_corrida.groupby('frac_lider').agg(['mean', 'std'])
    resumen.insert(0, 'percepción', percepcion)
    tablas.append(resumen)
lideres = pd.concat(tablas)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, kpi, titulo in [(axs[0], 'precision_rumbo', 'Precisión de rumbo'),
                        (axs[1], 'frac_grupo_mayor', 'Fracción de aves en el grupo mayor')]:
    for percepcion, datos in lideres.groupby('percepción', sort=False):
        ax.errorbar(100 * datos.index, datos[(kpi, 'mean')], yerr=datos[(kpi, 'std')],
                    marker='o', capsize=3, label=percepcion)
    ax.set(xlabel='Líderes (%)', title=titulo)
    ax.legend()
plt.tight_layout()
plt.show()
lideres.round(3)

**Resultados**

- **Con la percepción de la referencia se reproduce el resultado de Couzin et al. (2005):** pocos líderes guían a toda la parvada sin romperla. La precisión es de ~0,61 con 5% de líderes, ~0,89 con 10% y ~0,95 con 20%, y más del 98% de las aves sigue en el grupo mayor.
- **Con la percepción realista, la parvada también toma la dirección de los líderes** (precisión de ~0,56, ~0,78 y ~0,93), **pero se divide:** el grupo mayor reúne solo 45–55% de las aves, en 4 o 5 grupos. Como la polarización se mantiene alta (0,92–0,96), los grupos vuelan casi en la misma dirección, es decir, en paralelo. Sin líderes, el 94% de las aves sigue en el grupo mayor.
- **La variación entre réplicas es grande** con pocos líderes (desviación de la precisión de 0,3 a 0,56). La dirección preferida se elige al azar, así que en algunas réplicas la parvada tiene que girar hasta 180°, y en 100 s no siempre lo logra.

**Interpretación.** Con 7 vecinos, las aves cercanas a un líder lo siguen y se separan de las que no tienen un líder cerca; cada grupo termina orientado hacia la dirección preferida, pero no se mantienen juntos. Esto se parece al hallazgo de la sección 3.1: la interacción topológica transmite bien la **dirección**, pero mal la **cohesión** entre partes de la parvada. Es una interpretación que no se verifica aquí.

**Proporción elegida:** 10% de líderes, la menor fracción con la que, con percepción realista, la precisión supera 0,75. El efecto conjunto de la fracción de líderes y de `goal_weight` se analiza en la sección 6.

### 3.3 Mejora 3: energía

En las etapas anteriores, todas las aves vuelan siempre a 10 m/s y maniobran sin límite de esfuerzo. Esta mejora agrega un **estado interno**, la energía (de 0 a 1), que se gasta al volar, se recupera y limita la rapidez máxima del ave. Para eso, primero la rapidez se vuelve variable.

**Rapidez variable.** La rapidez deseada es la magnitud de la velocidad que piden las reglas sociales, más una tendencia a volver a la rapidez preferida:

rapidez deseada = |velocidad deseada| + `speed_relax` × (`speed_pref` − rapidez)

Las reglas ya contienen la información necesaria: la alineación iguala también la rapidez de los vecinos, y la cohesión acelera a las aves que se quedan atrás. La rapidez se recorta entre `v_min` = 6,9 m/s y la rapidez máxima que permite la energía, que con energía alta es `v_max` = 15 m/s. Estos extremos son las rapideces mínima y máxima de las parvadas medidas por Ballerini et al. (2008b); son rapideces de parvadas completas y se usan como aproximación de los límites de cada ave.

**Gasto por paso.**

costo = `cost_factor` × *p*(*v*) × *n*² × *s*(*k*)

| Componente | Definición | Fuente |
|---|---|---|
| *p*(*v*) | Potencia de vuelo relativa a la de 10 m/s, interpolada entre 10,4 W a 6,3 m/s y 14,9 W a 14,4 m/s: vale 0,84 a 6,3 m/s y 1,20 a 14,4 m/s | Dato: estorninos en túnel de viento (Ward et al., 2001) |
| *n*² | Factor de carga al cuadrado, *n*² = 1 + (*a*/*g*)², con aceleración lateral *a* = *v*·Δθ/Δ*t*. Volar recto cuesta 1 y un giro a 2 *g* cuesta 4 | Dato: duplicar la carga en un giro cuadruplica el esfuerzo (Usherwood et al., 2011); girar tiene un costo energético sustancial (Wilson et al., 2013) |
| *s*(*k*) | Multiplicador por **giros seguidos**: *s* = 1 + (*s*ₘₐₓ − 1)·(1 − e^(−(*k* − 1)/τ)), donde *k* cuenta los pasos seguidos con giro mayor que `turn_threshold` y vuelve a 0 con un paso sin giro. Un giro aislado no se encarece; con giros seguidos el costo crece y se satura en *s*ₘₐₓ | Supuesto razonado; no se encontraron datos sobre giros encadenados |
| `cost_factor` | 1 para adultos y líderes, 1,14 para juveniles (`cost_factor__juvenil`) | Dato: los juveniles de cigüeña gastan ~14% más en vuelo (Rotics et al., 2016); es otra especie, se usa como aproximación |

**Recuperación.** En cada paso:

energía ← energía + `fatigue_scale` × (*r*ᵢ − costo), recortada entre 0 y 1

La tasa de recuperación *r*ᵢ es propia de cada ave y constante: sigue una distribución normal truncada en 0, con media `recovery_mean` y coeficiente de variación `recovery_cv`. La tasa metabólica de las aves varía de forma consistente entre individuos (repetibilidad ~0,56; Versteegh et al., 2008); el valor del coeficiente de variación (10%) es un supuesto. El **umbral de recuperación** está implícito: cuando el costo de un paso es menor que *r*ᵢ, el ave recupera energía.

**Efecto de la energía.** Con energía menor que `e_crit`, la rapidez máxima baja linealmente hasta `v_min`:

*v*ₘₐₓ(*E*) = `v_min` + (`v_max` − `v_min`) × mín(1, *E* / `e_crit`)

Un ave agotada solo puede volar a 6,9 m/s. Si vuela más lento que su parvada, se rezaga.

**Calibración de la recuperación: el giro de fondo.** Dentro de una parvada formada, las aves corrigen su rumbo constantemente. Con la percepción realista, la mediana del giro es de solo 0,3° por paso, pero ~8% de los pasos son sacudidas aisladas al giro máximo: con umbral de 5°, una racha dura en promedio 1,3 pasos. Ese giro de fondo tiene un costo real: Usherwood et al. (2011) encontraron que volar en parvada cuesta más precisamente por las maniobras constantes. `recovery_mean` se fija igual al costo de fondo de un adulto (medido abajo), de modo que **un adulto típico queda en equilibrio**. La fatiga aparece entonces por razones concretas: el mayor gasto de los juveniles, una recuperación individual menor que la media, más maniobras que el promedio o volar más rápido.

**Escala de tiempo de la fatiga.** Una corrida dura 100 s simulados, y la fatiga real tarda mucho más. `fatigue_scale` comprime el tiempo: con 0,008, un juvenil típico pierde 0,008 × (1,44 − 1,26) ≈ 0,0014 de energía por paso y llega a `e_crit` hacia la mitad de la corrida. Es una simplificación declarada, y el parámetro se puede variar para estudiar escalas distintas.

| Parámetro | Valor por defecto (= Etapa 2) | Valor realista | Fuente |
|---|---|---|---|
| `v_min`, `v_max` | 1,0 m/paso (rapidez fija) | 0,69 y 1,5 m/paso (6,9 y 15 m/s) | Dato (Ballerini et al., 2008b) |
| `speed_relax` | 0 | 0,1 | Supuesto |
| `fatigue_scale` | 0 (sin gasto) | 0,008 | Supuesto: compresión del tiempo |
| `cost_factor`, `cost_factor__juvenil` | 1 | 1 y 1,14 | Dato (Rotics et al., 2016) |
| `turn_threshold` | — | 5° por paso | Elegido por encima del giro de fondo (90% de los pasos gira menos de 3,4°) |
| `streak_max`, `streak_tau` | — | 2 y 5 pasos | Supuesto |
| `recovery_mean` | — | 1,26 | Calibrado: costo de fondo de un adulto |
| `recovery_cv` | — | 0,10 | Supuesto (Versteegh et al., 2008, respalda que existan diferencias individuales) |
| `e_crit` | — | 0,3 | Supuesto |

#### Funciones de costo

In [ ]:
def costo_rapidez(v):
    """ Potencia de vuelo relativa a la de 10 m/s, interpolando las mediciones de
    Ward et al. (2001) en estorninos: 10,4 W a 6,3 m/s y 14,9 W a 14,4 m/s. """
    potencia = 10.4 + (14.9 - 10.4) * (v - 6.3) / (14.4 - 6.3)
    potencia_10 = 10.4 + (14.9 - 10.4) * (10 - 6.3) / (14.4 - 6.3)
    return potencia / potencia_10


def costo_giro(v, giro, dt):
    """ Factor de carga al cuadrado, n² = 1 + (a/g)², con aceleración lateral
    a = v·|giro|/dt. Volar recto cuesta 1; un giro a 2 g cuesta 4 (Usherwood et al., 2011). """
    a = v * abs(giro) / dt
    return 1 + (a / G) ** 2


def costo_racha(k, s_max, tau):
    """ Multiplicador por giros consecutivos: 1 para un giro aislado, crece con cada
    giro seguido y se satura en `s_max`. `k` es el número de giros seguidos. """
    return 1 + (s_max - 1) * (1 - np.exp(-max(k - 1, 0) / tau))

#### Agente

In [ ]:
class BoidEnergia(BoidRoles):
    """ Boid con energía (0 a 1) y rapidez variable.

    - La rapidez deseada sale de las reglas sociales (que incluyen igualar la
      velocidad de los vecinos) más una tendencia hacia `speed_pref`, y se recorta
      entre `v_min` y la rapidez máxima que permite la energía.
    - Cada paso gasta energía según la rapidez, los giros y los giros seguidos, por
      `cost_factor` (mayor en juveniles), y recupera `recovery_rate`, propio de cada ave.
    - Con energía menor que `e_crit`, la rapidez máxima baja linealmente hasta `v_min`.

    Con fatigue_scale = 0 y v_min = v_max = speed_pref se comporta igual que BoidRoles. """

    def setup(self):
        super().setup()
        self.energy = 1.0
        self.racha = 0  # Giros seguidos mayores que turn_threshold

    def v_max_actual(self):
        """ Rapidez máxima según la energía (m/paso). """
        v_min, v_max = self.p.get('v_min', self.p.speed_pref), self.p.get('v_max', self.p.speed_pref)
        fraccion = min(1.0, self.energy / self.p.e_crit) if self.p.get('e_crit', 0) > 0 else 1.0
        return v_min + (v_max - v_min) * fraccion

    def update_velocity(self):
        deseada = self.velocity + self.social_force()
        self.heading, self.turn = rotar_hacia(self.heading, deseada, self.max_turn())

        # Rapidez: la que piden las reglas, más una tendencia hacia la preferida
        objetivo = np.linalg.norm(deseada) + self.p.get('speed_relax', 0) * (self.p.speed_pref - self.speed)
        self.speed = float(np.clip(objetivo, self.p.get('v_min', self.p.speed_pref), self.v_max_actual()))
        self.velocity = self.heading * self.speed
        self.gastar_energia()

    def gastar_energia(self):
        escala = self.p.get('fatigue_scale', 0)
        if escala == 0:
            return
        self.racha = self.racha + 1 if abs(np.degrees(self.turn)) > self.p.turn_threshold else 0
        v = self.speed / self.p.dt  # m/s
        self.costo = (self.p.cost_factor * costo_rapidez(v) * costo_giro(v, self.turn, self.p.dt)
                      * costo_racha(self.racha, self.p.streak_max, self.p.streak_tau))
        self.energy = float(np.clip(self.energy + escala * (self.recovery_rate - self.costo), 0, 1))

#### Modelo y parámetros

In [ ]:
class ModeloEnergia(ModeloRoles):
    """ Modelo de la Etapa 3: roles + energía. La tasa de recuperación de cada ave
    sigue una distribución normal truncada en 0, con media `recovery_mean` y
    coeficiente de variación `recovery_cv`. """

    agent_class = BoidEnergia

    def setup(self):
        super().setup()
        media, cv = self.p.get('recovery_mean', 1.0), self.p.get('recovery_cv', 0.0)
        for a in self.agents:
            a.recovery_rate = max(0.0, self.nprandom.normal(media, cv * media)) if cv > 0 else media


class ModeloEnergiaConDatos(ModeloEnergia):
    """ Etapa 3 con el registro de la sección 3.0, más energía y rapidez por rol.
    Al final registra el rol, la energía y la distancia recorrida de cada ave. """

    def update(self):
        ModeloReferenciaConDatos.update(self)
        for rol in ROLES:
            aves = [a for a in self.agents if a.role == rol]
            if aves:
                self.record(f'energia_{rol}', np.mean([a.energy for a in aves]))
                self.record(f'rapidez_{rol}', np.mean([a.speed for a in aves]) / self.p.dt)
        for a in self.agents:
            a.distancia = getattr(a, 'distancia', 0.0) + (a.speed if self.t > 0 else 0.0)

    def end(self):
        self.agents.record(['role', 'energy', 'distancia'])

In [ ]:
parametros_energia = dict(
    parametros_roles,
    # Rapidez
    v_min=0.69,               # m/paso (6,9 m/s)
    v_max=1.5,                # m/paso (15 m/s)
    speed_relax=0.1,          # tendencia a volver a speed_pref
    # Gasto
    fatigue_scale=0.008,      # compresión del tiempo de fatiga
    cost_factor=1.0,
    cost_factor__juvenil=1.14,
    turn_threshold=5,         # grados por paso para contar en la racha
    streak_max=2.0,           # multiplicador máximo por giros seguidos
    streak_tau=5,             # giros seguidos para acercarse al máximo
    # Recuperación
    recovery_mean=1.26,       # calibrada en la sección 3.3
    recovery_cv=0.10,
    e_crit=0.3,               # energía bajo la cual baja la rapidez máxima
)

#### Calibración de la recuperación

Se mide el costo medio por paso de cada rol con la rapidez variable activada pero sin que la energía cambie (`fatigue_scale` casi 0), en 5 corridas, entre los pasos 200 y 1000.

In [ ]:
costos = {r: [] for r in ROLES}
for semilla in range(5):
    m = ModeloEnergia(dict(parametros_energia, fatigue_scale=1e-12, seed=semilla))
    m.sim_setup()
    while m.running:
        m.sim_step()
        if m.t > 200:
            for a in m.agents:
                costos[a.role].append(a.costo)
pd.Series({r: np.mean(c) for r, c in costos.items()}, name='costo medio por paso').round(3)

#### Dinámica de la energía

20 réplicas del modelo con los valores realistas.

In [ ]:
exp = ap.Experiment(ModeloEnergiaConDatos, ap.Sample(parametros_energia, n=1), iterations=20, record=True, randomize=True)
energia_exp = exp.run(n_jobs=-1, verbose=0)
serie = energia_exp.variables.ModeloEnergiaConDatos.reset_index().groupby('t').mean(numeric_only=True)

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for rol in ROLES:
    serie[f'energia_{rol}'].plot(ax=axs[0], label=rol)
    serie[f'rapidez_{rol}'].plot(ax=axs[1], label=rol)
axs[0].axhline(parametros_energia['e_crit'], color='gray', ls='--', label='e_crit')
axs[0].set(title='Energía media por rol', xlabel='Paso t')
axs[1].set(title='Rapidez media por rol (m/s)', xlabel='Paso t')
linea_d, = axs[2].plot(serie.index, serie['dist_vecino'], color='tab:blue', label='Distancia al vecino (m)')
banda = axs[2].axhspan(0.68, 1.51, color='green', alpha=0.15, label='Rango observado')
eje_grupos = axs[2].twinx()
linea_g, = eje_grupos.plot(serie.index, serie['n_grupos'], color='tab:orange', label='Número de grupos')
axs[2].set(title='Estructura de la parvada', xlabel='Paso t', ylabel='Distancia al vecino (m)')
eje_grupos.set_ylabel('Número de grupos')
axs[2].legend(handles=[linea_d, banda, linea_g], loc='lower right')
for ax in axs[:2]:
    ax.legend()
plt.tight_layout()
plt.show()

por_ave = energia_exp.variables.BoidEnergia.reset_index()
por_ave['fatigada'] = por_ave.energy < parametros_energia['e_crit']
por_ave.groupby('role')[['energy', 'fatigada', 'distancia']].mean().round(3)

**Resultados** (promedio de 20 réplicas, al final de la corrida)

- **Los juveniles se cansan y se rezagan.** Su energía media baja a ~0,30, el 66% termina bajo `e_crit`, vuelan más lento (~9,2 m/s frente a ~9,9 de los adultos, entre los pasos 500 y 1000) y recorren ~950 m frente a ~990 m.
- **Los adultos se mantienen**, con energía media de ~0,69. El 22% se fatiga: son los que tienen una recuperación menor que la media o maniobran más.
- **Liderar cuesta energía.** Los líderes vuelan más rápido (~11 a 12 m/s) para seguir su dirección preferida, recorren ~1150 m y el 43% termina fatigado. Este resultado no se programó directamente; surge de combinar la regla de los líderes con el gasto por rapidez.
- **La parvada se dispersa:** la distancia al vecino sube de ~1,1 m a ~1,8 m, por encima del rango observado, y aparecen 6 o 7 grupos. Las diferencias de rapidez entre aves estiran la parvada.

Si el resto de la parvada espera a las aves cansadas o las deja atrás se analiza en la sección 6, variando la cohesión, el alcance de visión y la escala de fatiga.

### 3.4 Modelo extendido completo

El modelo extendido es la Etapa 3 (`ModeloEnergia`) con los valores realistas de todas las etapas (`parametros_energia`). Cada parámetro nuevo tiene un valor por defecto que reproduce la etapa anterior, así que la referencia y cada etapa intermedia se obtienen del mismo modelo cambiando solo parámetros:

| Configuración | Cómo se obtiene |
|---|---|
| Referencia (Etapa 0) | `parametros_ref` |
| + Percepción (Etapa 1) | `parametros_percepcion` |
| + Roles (Etapa 2) | `parametros_roles` |
| + Energía (Etapa 3, modelo completo) | `parametros_energia` |

Esto permite comparar el efecto de cada mejora por separado en la sección 6.

In [ ]:
ModeloExtendido = ModeloEnergia
parametros_extendido = dict(parametros_energia)

ext = ModeloExtendido(parametros_extendido)
ext.run(display=False)

colores = {'lider': 'tab:red', 'adulto': 'tab:blue', 'juvenil': 'tab:orange'}
fig, ax = plt.subplots(figsize=(7, 7))
for rol, color in colores.items():
    aves = [a for a in ext.agents if a.role == rol]
    pos = np.array([a.pos for a in aves])
    rumbo = np.array([a.heading for a in aves])
    energia = np.array([a.energy for a in aves])
    ax.quiver(pos[:, 0], pos[:, 1], rumbo[:, 0], rumbo[:, 1], color=color, alpha=0.3 + 0.7 * energia,
              angles='xy', scale=40, width=0.004, label=rol)
ax.set(xlim=(0, ext.p.size), ylim=(0, ext.p.size), aspect='equal',
       title=f'Modelo extendido, t = {ext.t} (color: rol; opacidad: energía)')
ax.legend()
plt.show()

## 4. Calibración

### 4.1 Calibración del modelo de referencia

Con los valores originales del modelo base (`inner_radius` = 3, separación 0,1) pero en el espacio sin paredes, la distancia al vecino queda en ~3,1 m, el doble de lo observado. En la Parte I la distancia era ~1 m porque las paredes **comprimían** a la parvada, no por sus reglas.

Se calibran dos parámetros contra dos datos de 10 parvadas de estornino (Ballerini et al., 2008b):
- **Distancia media al vecino:** entre 0,68 y 1,51 m, con mediana de 1,05 m.
- **Distancia mínima:** casi ninguna ave a menos de 0,38 m (zona de exclusión de 0,19 m de radio, comparable a la envergadura de 0,4 m).

**Procedimiento:** 20 réplicas por valor, 1000 pasos cada una, y métricas promediadas entre los pasos 500 y 1000, cuando la parvada ya se formó.
1. Barrido de `inner_radius` con la separación original (0,1).
2. Con `inner_radius` fijo, barrido de `separation_strength`.

In [ ]:
def calibrar(cambios):
    """ Corre 20 réplicas por configuración y promedia las métricas de los pasos 500 a 1000. """
    muestra = ap.Sample(dict(parametros_ref, **cambios))
    exp = ap.Experiment(ModeloReferenciaConDatos, muestra, iterations=20, record=True, randomize=True)
    variables = exp.run(n_jobs=-1, verbose=0).arrange_variables()
    barrido = [k for k, v in cambios.items() if isinstance(v, ap.Values)][0]
    por_corrida = (variables[variables.t >= 500]
                   .groupby([barrido, 'iteration'])
                   [['dist_vecino', 'frac_colision', 'polarizacion', 'frac_grupo_mayor', 'n_grupos']]
                   .mean())
    return variables, por_corrida

var_r, cal_r = calibrar({'inner_radius': ap.Values(0.5, 0.75, 1.0, 1.5, 2.0, 3.0),
                          'separation_strength': 0.1})
cal_r.groupby('inner_radius').agg(['mean', 'std']).round(3)

In [ ]:
var_s, cal_s = calibrar({'inner_radius': 1.0,
                          'separation_strength': ap.Values(0.1, 0.2, 0.4, 0.8)})
cal_s.groupby('separation_strength').agg(['mean', 'std']).round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, cal, x, titulo in [(axs[0], cal_r, 'inner_radius', 'Paso 1: inner_radius (separación = 0,1)'),
                           (axs[1], cal_s, 'separation_strength', 'Paso 2: separación (inner_radius = 1 m)')]:
    resumen = cal.groupby(x).agg(['mean', 'std'])
    ax.errorbar(resumen.index, resumen[('dist_vecino', 'mean')], yerr=resumen[('dist_vecino', 'std')],
                marker='o', capsize=3, label='Distancia al vecino (m)')
    ax.axhspan(0.68, 1.51, color='green', alpha=0.15, label='Rango observado (0,68–1,51 m)')
    ax2 = ax.twinx()
    ax2.plot(resumen.index, 100 * resumen[('frac_colision', 'mean')], 'r--s', label='Aves en colisión (%)')
    ax2.set_ylabel('Aves a menos de 0,38 m (%)', color='r')
    ax.set(xlabel=x, ylabel='Distancia al vecino (m)', title=titulo)
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

elegida = var_s[var_s.separation_strength == parametros_ref['separation_strength']]
limite = parametros_ref['size'] - 2 * parametros_ref['outer_radius']
print(f"Extensión máxima del grupo mayor en las 20 réplicas: {elegida.extension.max():.1f} m "
      f"(límite: {limite} m) → {'cumple' if elegida.extension.max() < limite else 'NO cumple'}")

**Resultados**

- **Paso 1.** La distancia al vecino crece casi en proporción a `inner_radius`: es ~0,51 m con 0,5 m y ~3,07 m con 3 m. Con `inner_radius` = 1 m queda en 0,91 m, dentro del rango observado. Sin embargo, ~7% de las aves queda a menos de 0,38 m de otra. Esto viene de una limitación del modelo base (sección 1, pregunta 4c): la separación empuja más cuanto **más lejos** está el vecino, así que es débil justo cuando dos aves están muy cerca.
- **Paso 2.** Con separación de 0,2 a 0,8, la fracción en colisión baja a 0–0,2% y la distancia al vecino sube a 1,02–1,07 m.

**Valores elegidos:** `inner_radius` = 1 m y `separation_strength` = 0,4. Con ellos, la distancia al vecino es **1,055 m**, prácticamente la mediana observada (1,05 m), y la fracción en colisión es ~0,1%.

**Tamaño del torus:** la extensión máxima del grupo mayor queda muy por debajo del límite de 180 m (200 m menos dos radios de percepción), así que la parvada no se alcanza a sí misma.

**Para la validación:** la parvada se mantiene como un solo grupo en las 20 réplicas. Su polarización, 0,995, queda algo por encima del rango observado (0,96 ± 0,03, Cavagna et al., 2010): el modelo es más ordenado que una parvada real porque no tiene ruido en la percepción ni en el movimiento (candidata 7 de la sección 2.1).

### 4.2 Análisis de sensibilidad

El análisis de sensibilidad varía 13 parámetros del modelo completo, uno a la vez, un 20% hacia abajo y hacia arriba. Usa la infraestructura de experimentos con réplicas de la sección 6, así que se presenta ahí como el experimento E6 (sección 6.6).

## 5. Recolección de datos (KPIs)

Cada KPI mide el efecto de un mecanismo concreto del modelo. Se registran con `record()` en cada paso y quedan en el `ap.DataDict` de salida. Al final de cada corrida se registran también valores acumulados de cada ave, para analizar cómo se distribuyen entre individuos.

El registro está separado de los modelos: la clase `RegistroKPI` se combina con cualquier etapa de la Parte III sin modificarla. Los KPIs de roles, dirección preferida y energía se activan solos cuando el modelo tiene esos mecanismos.

**KPIs por paso**

| KPI | Definición | Mecanismo que mide | Desde |
|---|---|---|---|
| `polarizacion` | Magnitud del rumbo promedio (0 a 1) | Alineación | Etapa 0 |
| `dist_vecino` | Distancia media al vecino más cercano (m) | Equilibrio entre cohesión y separación; se compara con 0,68–1,51 m | Etapa 0 |
| `frac_colision` | Fracción de aves con una vecina a menos de 0,38 m | Realismo de la separación | Etapa 0 |
| `n_grupos`, `frac_grupo_mayor` | Grupos conectados por vecinas a menos de `outer_radius`, y fracción de aves en el mayor | Cohesión y fragmentación | Etapa 0 |
| `extension` | Mayor distancia entre dos aves del grupo mayor (m) | Tamaño de la parvada; criterio del torus | Etapa 0 |
| `frac_aislados` | Fracción de aves sin otra a menos de `outer_radius` | Aves rezagadas o separadas | Etapa 0 |
| `rapidez_media`, `rapidez_std` | Rapidez media y su dispersión (m/s) | Si el grupo se adapta a las aves lentas | Etapa 3 (antes es constante) |
| `precision_rumbo` | Promedio del coseno entre el rumbo de cada ave y la dirección preferida de los líderes (−1 a 1) | Efecto de los líderes | Etapa 2 |
| `energia_media`, `energia_min`, `energia_p10` | Media, mínimo y percentil 10 de la energía (0 a 1) | Fatiga | Etapa 3 |
| `frac_fatigados` | Fracción de aves con energía menor que `e_crit` | Fatiga | Etapa 3 |
| `fatigados_en_grupo_mayor`, `descansados_en_grupo_mayor` | Fracción de las aves fatigadas, y de las descansadas, que están en el grupo mayor | Si la parvada conserva a las aves cansadas | Etapa 3 |
| `rapidez_fatigados`, `rapidez_descansados` | Rapidez media de cada grupo (m/s) | Si las aves descansadas frenan para ir al paso de las cansadas | Etapa 3 |
| `polarizacion_<rol>`, `aislados_<rol>`, `rapidez_<rol>`, `precision_<rol>`, `energia_<rol>` | Los mismos KPIs, calculados dentro de cada rol | Diferencias entre roles | Etapa 2 |

**Valores al final de cada corrida**

| Variable | Nivel | Definición |
|---|---|---|
| `distancia_total` | Ave | Metros recorridos |
| `giro_total` | Ave | Grados girados, en valor absoluto |
| `pasos_aislado` | Ave | Pasos sin otra ave a menos de `outer_radius` |
| `role`, `energy` | Ave | Rol y energía final (Etapas 2 y 3) |

Para activarse, las etapas siguientes definen estos atributos: `role` y `energy` en el agente, `goal_dir` en el modelo, y el parámetro `e_crit`.

In [ ]:
def indicadores(m):
    """ KPIs del paso actual y la máscara de aves aisladas.

    Los indicadores de estructura y rapidez se calculan siempre. Los de energía,
    dirección preferida y roles se agregan solo si el modelo tiene esos
    mecanismos: agentes con `energy` (Etapa 3), modelo con `goal_dir` (Etapa 2)
    y agentes con `role` (Etapa 2). """
    d = distancias_toroidales(m)
    np.fill_diagonal(d, np.inf)
    kpi = metricas_parvada(m, d)

    agentes = list(m.agents)
    rumbos = np.array([a.heading for a in agentes])
    rapidez = np.array([a.speed for a in agentes]) / m.p.dt  # m/s
    aislado = d.min(axis=1) >= m.p.outer_radius
    kpi['frac_aislados'] = aislado.mean()
    kpi['rapidez_media'] = rapidez.mean()
    kpi['rapidez_std'] = rapidez.std()

    tiene_energia = hasattr(agentes[0], 'energy')
    if tiene_energia:
        energia = np.array([a.energy for a in agentes])
        kpi['energia_media'] = energia.mean()
        kpi['energia_min'] = energia.min()
        kpi['energia_p10'] = np.percentile(energia, 10)
        fatigado = energia < m.p.get('e_crit', 0)
        kpi['frac_fatigados'] = fatigado.mean()
        # ¿Las aves fatigadas siguen con la parvada? Fracción de cada grupo en el grupo mayor
        _, etiquetas = connected_components(csr_matrix(d < m.p.outer_radius), directed=False)
        en_mayor = etiquetas == np.bincount(etiquetas).argmax()
        kpi['fatigados_en_grupo_mayor'] = en_mayor[fatigado].mean() if fatigado.any() else np.nan
        kpi['descansados_en_grupo_mayor'] = en_mayor[~fatigado].mean() if (~fatigado).any() else np.nan
        kpi['rapidez_fatigados'] = rapidez[fatigado].mean() if fatigado.any() else np.nan
        kpi['rapidez_descansados'] = rapidez[~fatigado].mean() if (~fatigado).any() else np.nan

    tiene_direccion = hasattr(m, 'goal_dir')
    if tiene_direccion:
        precision = rumbos @ m.goal_dir  # Coseno entre cada rumbo y la dirección preferida
        kpi['precision_rumbo'] = precision.mean()

    if hasattr(agentes[0], 'role'):
        roles = np.array([a.role for a in agentes])
        for r in np.unique(roles):
            s = roles == r
            kpi[f'polarizacion_{r}'] = np.linalg.norm(rumbos[s].mean(axis=0))
            kpi[f'aislados_{r}'] = aislado[s].mean()
            kpi[f'rapidez_{r}'] = rapidez[s].mean()
            if tiene_energia:
                kpi[f'energia_{r}'] = energia[s].mean()
            if tiene_direccion:
                kpi[f'precision_{r}'] = precision[s].mean()

    return kpi, aislado

In [ ]:
class RegistroKPI:
    """ Agrega a un modelo el registro de KPIs en cada paso y, al final, los valores
    acumulados de cada agente. Se combina con cualquier modelo de la Parte III:

        class ModeloXKPI(RegistroKPI, ModeloX):
            pass
    """

    def setup(self):
        super().setup()
        for a in self.agents:
            a.distancia_total = 0.0  # m recorridos
            a.giro_total = 0.0       # grados girados, en valor absoluto
            a.pasos_aislado = 0      # pasos sin otra ave a menos de outer_radius

    def update(self):
        super().update()
        kpi, aislado = indicadores(self)
        for nombre, valor in kpi.items():
            self.record(nombre, valor)
        if self.t > 0:
            for a, solo in zip(self.agents, aislado):
                a.distancia_total += a.speed
                a.giro_total += abs(np.degrees(a.turn))
                a.pasos_aislado += int(solo)

    def end(self):
        super().end()
        por_agente = ['distancia_total', 'giro_total', 'pasos_aislado']
        por_agente += [v for v in ('role', 'energy') if hasattr(self.agents[0], v)]
        self.agents.record(por_agente)


class ModeloReferenciaKPI(RegistroKPI, ModeloReferencia):
    """ Modelo de referencia con registro de KPIs. """


class ModeloPercepcionKPI(RegistroKPI, ModeloPercepcion):
    """ Etapa 1 con registro de KPIs. """


class ModeloRolesKPI(RegistroKPI, ModeloRoles):
    """ Etapa 2 con registro de KPIs. """


class ModeloEnergiaKPI(RegistroKPI, ModeloEnergia):
    """ Etapa 3 con registro de KPIs. """

#### Ejemplo: modelo de referencia

In [ ]:
salida_kpi = ModeloReferenciaKPI(parametros_ref).run(display=False)
kpi_ref = salida_kpi.variables.ModeloReferenciaKPI
por_ave = salida_kpi.variables.BoidReferencia

columnas = ['polarizacion', 'dist_vecino', 'frac_colision', 'n_grupos', 'frac_grupo_mayor', 'frac_aislados']
axs = kpi_ref[columnas].plot(subplots=True, layout=(2, 3), figsize=(15, 6), legend=False)
for ax, nombre in zip(axs.flat, columnas):
    ax.set_title(nombre)
    ax.set_xlabel('Paso t')
axs.flat[1].axhspan(0.68, 1.51, color='green', alpha=0.15)
plt.tight_layout()
plt.show()

por_ave.describe().round(1)

En el modelo de referencia la rapidez es constante (10 m/s), así que todas las aves recorren 1000 m. Estos indicadores empiezan a variar en la Etapa 3. El giro acumulado, en cambio, sí varía entre aves: mide cuánto maniobra cada una y es la base del gasto por giros de la Etapa 3.

## 6. Experimentos

**Diseño común**
- Modelo: el modelo completo de la sección 3.4. Cada configuración apaga o cambia mecanismos solo mediante parámetros.
- 1000 pasos (100 s de vuelo) y **20 réplicas** por configuración. Todas las configuraciones usan las mismas 20 semillas, para que las diferencias entre ellas no se deban al azar de la condición inicial.
- Los KPIs de cada corrida se promedian entre los pasos 500 y 1000, cuando ya pasó el arranque; los valores por ave son los del final de la corrida.
- Se reporta la media entre réplicas y su **intervalo de confianza del 95%** (t de Student con 19 grados de libertad).

| Experimento | Pregunta | Configuraciones |
|---|---|---|
| E1 | ¿Qué efecto tiene cada mejora? | 6 |
| E2 | ¿Cómo influyen el número de vecinos y el punto ciego? | 4 × 3 |
| E3 | ¿Cuántos líderes hacen falta, y con qué peso? | 4 × 4 |
| E4 | ¿La parvada espera a las aves cansadas? | 4 × 3 |
| E5 | ¿Qué efecto tiene la variación individual en la recuperación? | 3 |
| E6 | ¿Qué parámetros influyen más en los resultados? (sensibilidad) | 27 |

**Resultados guardados.** Las 1520 corridas tardan ~30 minutos en una computadora de 24 núcleos. Sus resúmenes se guardan en la carpeta `resultados/` y el notebook los carga si existen. Con `RECALCULAR = True` se vuelven a correr; como las semillas son fijas, el resultado es el mismo.

#### Infraestructura

In [ ]:
import os
from scipy import stats

CARPETA_RESULTADOS = 'resultados'
RECALCULAR = False  # True: vuelve a correr los experimentos aunque haya resultados guardados
REPLICAS = 20
INICIO_PROMEDIO = 500  # Los KPIs por corrida se promedian entre este paso y el final

SERIES = ['polarizacion', 'dist_vecino', 'n_grupos', 'frac_grupo_mayor', 'frac_aislados', 'rapidez_media',
          'precision_rumbo', 'energia_media', 'frac_fatigados', 'fatigados_en_grupo_mayor', 'descansados_en_grupo_mayor']


def correr_experimento(nombre, configuraciones, Modelo=None, replicas=None):
    """ Corre cada configuración con `replicas` réplicas, o carga el resultado guardado.

    `configuraciones` es un diccionario {etiqueta: parámetros}; los parámetros pueden
    incluir `ap.Values` para barrer valores. Todas las configuraciones usan las mismas
    semillas por réplica, para que las diferencias no se deban al azar de la condición
    inicial. Devuelve:
    - `resumen`: una fila por corrida, con los KPIs promediados entre INICIO_PROMEDIO y
      el final, y los valores finales por ave promediados por rol (prefijo `final_`).
    - `series`: la media por paso de cada configuración. """
    Modelo = Modelo or ModeloEnergiaKPI
    replicas = replicas or REPLICAS
    ruta = {k: os.path.join(CARPETA_RESULTADOS, f'{nombre}_{k}.csv') for k in ('resumen', 'series')}
    if not RECALCULAR and all(os.path.exists(r) for r in ruta.values()):
        return pd.read_csv(ruta['resumen']), pd.read_csv(ruta['series'])

    resumenes, series = [], []
    for etiqueta, parametros in configuraciones.items():
        barridos = [k for k, v in parametros.items() if isinstance(v, ap.Values)]
        muestra = ap.Sample(parametros) if barridos else ap.Sample(parametros, n=1)
        salida = ap.Experiment(Modelo, muestra, iterations=replicas, record=True, randomize=True).run(n_jobs=-1, verbose=0)
        valores = salida.parameters.sample.reset_index()[['sample_id'] + barridos] if barridos else None

        kpi = salida.variables[Modelo.__name__].reset_index()
        aves = salida.variables[Modelo.agent_class.__name__].reset_index()
        for tabla in (kpi, aves):
            if 'sample_id' not in tabla:
                tabla['sample_id'] = 0  # Sin barrido, agentpy no incluye sample_id
        por_corrida = kpi[kpi.t >= INICIO_PROMEDIO].drop(columns='t').groupby(['sample_id', 'iteration']).mean()
        if 'role' not in aves:
            aves['role'] = 'todas'
        if 'energy' in aves:
            aves['fatigada'] = aves['energy'] < parametros.get('e_crit', 0)
        columnas = [c for c in ('energy', 'fatigada', 'distancia_total', 'giro_total', 'pasos_aislado') if c in aves]
        por_rol = aves.groupby(['sample_id', 'iteration', 'role'])[columnas].mean().unstack('role')
        por_rol.columns = [f'final_{c}_{r}' for c, r in por_rol.columns]
        resumen = por_corrida.join(por_rol).reset_index()
        resumen.insert(0, 'config', etiqueta)

        serie = kpi.groupby(['sample_id', 't'])[[c for c in SERIES if c in kpi]].mean().reset_index()
        serie.insert(0, 'config', etiqueta)
        if barridos:
            resumen = resumen.merge(valores, on='sample_id')
            serie = serie.merge(valores, on='sample_id')
        resumenes.append(resumen)
        series.append(serie)

    resumen, serie = pd.concat(resumenes, ignore_index=True), pd.concat(series, ignore_index=True)
    os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
    resumen.to_csv(ruta['resumen'], index=False)
    serie.to_csv(ruta['series'], index=False)
    return resumen, serie


def media_ic(resumen, por, columnas):
    """ Media e intervalo de confianza del 95% (t de Student) entre réplicas. """
    g = resumen.groupby(por, sort=False)[columnas]
    media, n = g.mean(), g.count()
    mitad = g.std() / np.sqrt(n) * stats.t.ppf(0.975, n - 1)
    return media, mitad

In [ ]:
RANGO_DIST = (0.68, 1.51)   # Distancia al vecino observada (Ballerini et al., 2008b)
RANGO_POL = (0.93, 0.99)    # Polarización observada, 0,96 ± 0,03 (Cavagna et al., 2010)


def barras_ic(resumen, por, kpis, titulos, bandas=None):
    """ Barras con la media de cada configuración y su intervalo de confianza del 95%. """
    media, ic = media_ic(resumen, por, kpis)
    fig, axs = plt.subplots(1, len(kpis), figsize=(4.3 * len(kpis), 4.2))
    for ax, k, titulo in zip(np.atleast_1d(axs), kpis, titulos):
        ax.bar(range(len(media)), media[k], yerr=ic[k], capsize=3, color='tab:blue', alpha=0.8)
        if bandas and k in bandas:
            ax.axhspan(*bandas[k], color='green', alpha=0.15, label='Rango observado')
            ax.legend(loc='lower right')
        ax.set_xticks(range(len(media)), media.index, rotation=40, ha='right')
        ax.set_title(titulo)
    plt.tight_layout()
    plt.show()
    return media, ic


def lineas_ic(resumen, x, grupo, kpis, titulos, etiqueta_grupo, bandas=None):
    """ Una línea por valor de `grupo`, con la media en función de `x` y su IC del 95%. """
    fig, axs = plt.subplots(1, len(kpis), figsize=(4.6 * len(kpis), 4.2))
    media, ic = media_ic(resumen, [grupo, x], kpis)
    for ax, k, titulo in zip(np.atleast_1d(axs), kpis, titulos):
        for valor in media.index.get_level_values(grupo).unique():
            m, e = media.loc[valor, k], ic.loc[valor, k]
            ax.errorbar(m.index, m.values, yerr=e.values, marker='o', capsize=3, label=f'{etiqueta_grupo} = {valor:g}')
        if bandas and k in bandas:
            ax.axhspan(*bandas[k], color='green', alpha=0.15)
        ax.set(title=titulo, xlabel=x)
    np.atleast_1d(axs)[0].legend()
    plt.tight_layout()
    plt.show()
    return media, ic

### 6.1 E1: efecto de cada mejora

Las mejoras se agregan una a una sobre la referencia (filas 2 a 4). Además, dos configuraciones aplican una sola mejora con la percepción de la referencia: "Solo roles" y "Solo energía" (esta última con juveniles y sin líderes).

In [ ]:
percepcion_referencia = dict(n_neighbors=None, vision_angle=360, vision_radius=10)

configs_e1 = {
    'Referencia': parametros_ref,
    '+ Percepción': parametros_percepcion,
    '+ Roles': parametros_roles,
    '+ Energía (completo)': parametros_energia,
    'Solo roles': dict(parametros_roles, **percepcion_referencia),
    'Solo energía': dict(parametros_energia, **percepcion_referencia, frac_lider=0),
}
e1, serie_e1 = correr_experimento('e1_ablacion', configs_e1)

In [ ]:
media_e1, ic_e1 = barras_ic(e1, 'config', ['frac_grupo_mayor', 'polarizacion', 'dist_vecino', 'energia_media'],
                            ['Fracción en el grupo mayor', 'Polarización', 'Distancia al vecino (m)', 'Energía media'],
                            bandas={'polarizacion': RANGO_POL, 'dist_vecino': RANGO_DIST})
columnas = ['final_fatigada_adulto', 'final_fatigada_juvenil', 'final_fatigada_lider',
            'final_pasos_aislado_adulto', 'final_pasos_aislado_juvenil', 'rapidez_media', 'precision_rumbo']
media_ic(e1, 'config', columnas)[0].round(3)

**Resultados**

- **Percepción realista:** la parvada sigue casi unida (94% de las aves en el grupo mayor, frente a 100% en la referencia) y un poco más espaciada (1,14 m frente a 1,06 m entre vecinos).
- **Roles:** con la percepción realista, los líderes dividen la parvada: el grupo mayor baja a 62% y la polarización a 0,95. Con la percepción de la referencia ("Solo roles"), los mismos líderes la guían (precisión 0,85) sin dividirla (99,7%). **La fragmentación surge de combinar líderes con interacción topológica**, no de los líderes solos.
- **Energía (modelo completo):** la parvada se dispersa (distancia al vecino 1,85 m, por encima del rango observado) y el grupo mayor baja a 57%. La energía media termina en 0,50. El 66% de los juveniles termina fatigado, contra 22% de los adultos y 43% de los líderes; los juveniles pasan en promedio 12 pasos aislados, contra 1 de los adultos.
- **"Solo energía" agota mucho más a las aves** (energía media 0,19, rapidez 7,9 m/s). La recuperación se calibró con el giro de fondo de la percepción realista (sección 3.3), y con la percepción de la referencia las aves giran más en cada paso (~3,6° frente a ~1,4°), así que gastan más de lo que recuperan. **La calibración de la energía depende de la percepción**: la combinación de mecanismos no es aditiva.

### 6.2 E2: número de vecinos y punto ciego

Sobre la Etapa 1 (percepción realista, sin roles ni energía), se varían el número de vecinos (3, 7, 15 y 30) y el ángulo de visión (360°, 300° y 240°).

In [ ]:
configs_e2 = {'Percepción': dict(parametros_percepcion, n_neighbors=ap.Values(3, 7, 15, 30),
                                 vision_angle=ap.Values(360, 300, 240))}
e2, serie_e2 = correr_experimento('e2_percepcion', configs_e2)

In [ ]:
media_e2, ic_e2 = lineas_ic(e2, 'n_neighbors', 'vision_angle', ['frac_grupo_mayor', 'polarizacion', 'dist_vecino'],
                            ['Fracción en el grupo mayor', 'Polarización', 'Distancia al vecino (m)'],
                            'ángulo de visión', bandas={'polarizacion': RANGO_POL, 'dist_vecino': RANGO_DIST})

**Resultados**

- **Más vecinos, parvada más unida.** Sin punto ciego (360°), el grupo mayor reúne 48% de las aves con 3 vecinos, 73% con 7, 98% con 15 y 100% con 30. La distancia al vecino baja ligeramente (de 1,21 a 1,10 m).
- **El punto ciego también une a la parvada.** Con 7 vecinos, el grupo mayor pasa de 73% (360°) a 86% (300°) y a 98% (240°). Confirma lo observado en la sección 3.1: al ignorar a las aves de atrás, cada ave sigue a las que van delante y a los lados.
- **La polarización se mantiene alta (≥ 0,985) en todas las configuraciones.** Los grupos que se separan siguen volando en la misma dirección.
- **La configuración realista (7 vecinos, 300°) está en la zona de transición:** 86% ± 8% en el grupo mayor. Un cambio pequeño del ángulo o del número de vecinos mueve bastante la cohesión, lo que coincide con el análisis de sensibilidad (E6). En la sección 3.1, con otras semillas, la misma configuración dio 94%, dentro de ese intervalo.

### 6.3 E3: proporción de líderes y peso de la dirección preferida

Sobre el modelo completo, se varían la fracción de líderes (2%, 5%, 10% y 20%) y `goal_weight` (0,02, 0,05, 0,1 y 0,2).

In [ ]:
configs_e3 = {'Liderazgo': dict(parametros_energia, frac_lider=ap.Values(0.02, 0.05, 0.1, 0.2),
                                goal_weight=ap.Values(0.02, 0.05, 0.1, 0.2))}
e3, serie_e3 = correr_experimento('e3_liderazgo', configs_e3)

In [ ]:
media_e3, ic_e3 = lineas_ic(e3, 'frac_lider', 'goal_weight', ['precision_rumbo', 'frac_grupo_mayor', 'final_fatigada_lider'],
                            ['Precisión de rumbo', 'Fracción en el grupo mayor', 'Líderes fatigados al final'],
                            'goal_weight')

**Resultados**

- **Más líderes, más precisión.** Con `goal_weight` = 0,05, la precisión de rumbo es 0,24 con 2% de líderes, 0,38 con 5%, 0,58 con 10% y 0,91 con 20%. Coincide cualitativamente con Couzin et al. (2005): una minoría informada orienta al grupo. Pero aquí hace falta una minoría mayor, porque la parvada se divide.
- **Más líderes, menos cohesión.** El grupo mayor baja de ~75–80% con 2% de líderes a ~50–60% con 20%.
- **Un peso alto con pocos líderes es contraproducente.** Con 2% de líderes y peso 0,2, la precisión es negativa (−0,12): los líderes se separan de la parvada en lugar de arrastrarla.
- **Liderar cansa.** La fracción de líderes fatigados sube con el peso y con la proporción de líderes, de ~20% a ~58%.
- **La energía reduce la eficacia de los líderes.** Con 10% de líderes y peso 0,05, la precisión es 0,58 en el modelo completo, frente a 0,78 sin energía (sección 3.2). Los líderes se cansan, frenan y pierden influencia.
- La precisión varía mucho entre réplicas (intervalos de hasta ±0,36), porque depende de cuánto tenga que girar la parvada hacia la dirección preferida, que es aleatoria.

### 6.4 E4: ¿la parvada espera a las aves cansadas?

Sobre el modelo completo, se varían la cohesión (0,0025, 0,005, 0,01 y 0,02) y la escala de fatiga (0,004, 0,008 y 0,016). Si la parvada "espera", las aves descansadas deberían frenar para ir al paso de las cansadas, y los juveniles deberían quedar menos tiempo aislados.

In [ ]:
configs_e4 = {'Rezagados': dict(parametros_energia, cohesion_strength=ap.Values(0.0025, 0.005, 0.01, 0.02),
                                fatigue_scale=ap.Values(0.004, 0.008, 0.016))}
e4, serie_e4 = correr_experimento('e4_rezagados', configs_e4)

In [ ]:
media_e4, ic_e4 = lineas_ic(e4, 'cohesion_strength', 'fatigue_scale',
                            ['rapidez_descansados', 'final_pasos_aislado_juvenil', 'frac_fatigados', 'dist_vecino'],
                            ['Rapidez de las aves descansadas (m/s)', 'Pasos aislado por juvenil', 'Fracción de aves fatigadas',
                             'Distancia al vecino (m)'],
                            'fatigue_scale', bandas={'dist_vecino': RANGO_DIST})
media_ic(e4, ['fatigue_scale', 'cohesion_strength'], ['fatigados_en_grupo_mayor', 'descansados_en_grupo_mayor'])[0].round(3)

**Resultados**

- **Con cohesión débil, la parvada no espera.** Con cohesión 0,0025 y la escala de fatiga base (0,008), las aves descansadas siguen a ~10,3 m/s y cada juvenil pasa ~19 pasos aislado. Con fatiga más rápida (0,016), ~29 pasos.
- **Con cohesión fuerte, la parvada sí espera.** Con cohesión 0,02, las aves descansadas bajan a ~8,2 m/s para ir al paso de las cansadas, los juveniles casi no quedan aislados (~0,5 pasos) y la distancia al vecino vuelve al rango observado (1,13 m).
- **Pero esperar cuesta.** Con cohesión fuerte, casi toda la parvada termina fatigada: 18% de las aves con cohesión 0,0025 y 99% con 0,02. La cohesión obliga a más maniobras para mantenerse juntas, y los giros gastan energía.
- **La fatiga, por sí sola, no expulsa a las aves de la parvada:** la fracción de aves fatigadas en el grupo mayor es parecida a la de las descansadas (~0,4 a 0,6 en ambos casos). Las aves cansadas se rezagan sobre todo cuando la cohesión es débil.

**Respuesta:** depende de la cohesión. Hay un compromiso entre mantener unida a la parvada y el gasto de energía de todas sus aves.

### 6.5 E5: variación individual en la recuperación

Sobre el modelo completo, se varía el coeficiente de variación de la tasa de recuperación (0%, 10% y 20%).

In [ ]:
configs_e5 = {'Recuperación': dict(parametros_energia, recovery_cv=ap.Values(0.0, 0.1, 0.2))}
e5, serie_e5 = correr_experimento('e5_recuperacion', configs_e5)

In [ ]:
media_e5, ic_e5 = media_ic(e5, 'recovery_cv', ['energia_media', 'energia_p10', 'final_fatigada_adulto',
                                               'final_fatigada_juvenil', 'frac_grupo_mayor'])
media_e5.round(3).astype(str) + ' ± ' + ic_e5.round(3).astype(str)

**Resultados**

- **La energía media no cambia** (~0,49 a 0,51), pero la distribución se ensancha: el percentil 10 de la energía baja de 0,075 a casi 0.
- **La variación individual redistribuye la fatiga entre roles.** Sin variación, se fatiga el 9% de los adultos y el 86% de los juveniles; con 20% de variación, el 34% de los adultos y el 60% de los juveniles. Con variación, algunos adultos recuperan menos que el promedio y algunos juveniles más.
- **Más variación, menos cohesión:** el grupo mayor baja de 63% a 47%.

### 6.6 E6: análisis de sensibilidad

Sobre el modelo completo, cada parámetro se cambia un 20% hacia abajo y hacia arriba, uno a la vez. Para `n_neighbors` se usan 6 y 8 vecinos (±14%), porque debe ser entero. Las barras muestran el cambio de cada KPI respecto a la configuración base; la franja gris es el intervalo de confianza del 95% de la base: un cambio dentro de ella no se distingue de la variación entre réplicas.

In [ ]:
variaciones = {
    'inner_radius': None, 'separation_strength': None, 'cohesion_strength': None, 'alignment_strength': None,
    'n_neighbors': (6, 8), 'vision_angle': None, 'goal_weight': None, 'frac_lider': None, 'frac_juvenil': None,
    'fatigue_scale': None, 'recovery_mean': None, 'e_crit': None, 'max_load_factor': None,
}
configs_e6 = {'Base': parametros_energia}
for parametro, valores in variaciones.items():
    base = parametros_energia[parametro]
    bajo, alto = valores or (base * 0.8, base * 1.2)
    configs_e6[f'{parametro} −20%'] = dict(parametros_energia, **{parametro: bajo})
    configs_e6[f'{parametro} +20%'] = dict(parametros_energia, **{parametro: alto})
e6, serie_e6 = correr_experimento('e6_sensibilidad', configs_e6)

In [ ]:
kpis_e6 = ['frac_grupo_mayor', 'precision_rumbo', 'energia_media', 'dist_vecino']
titulos_e6 = ['Fracción en el grupo mayor', 'Precisión de rumbo', 'Energía media', 'Distancia al vecino (m)']
media_e6, _ = media_ic(e6, 'config', kpis_e6)
_, ic_base = media_ic(e6[e6.config == 'Base'], 'config', kpis_e6)
nombres = list(variaciones)
y = np.arange(len(nombres))

fig, axs = plt.subplots(1, len(kpis_e6), figsize=(18, 6), sharey=True)
for ax, k, titulo in zip(axs, kpis_e6, titulos_e6):
    base = media_e6.loc['Base', k]
    ax.barh(y - 0.2, [media_e6.loc[f'{p} −20%', k] - base for p in nombres], height=0.4, label='−20%')
    ax.barh(y + 0.2, [media_e6.loc[f'{p} +20%', k] - base for p in nombres], height=0.4, label='+20%')
    ax.axvspan(-ic_base.loc['Base', k], ic_base.loc['Base', k], color='gray', alpha=0.25, label='IC 95% de la base')
    ax.axvline(0, color='black', lw=0.8)
    ax.set(title=titulo, xlabel='Cambio respecto a la base')
axs[0].set_yticks(y, nombres)
axs[0].invert_yaxis()
fig.legend(*axs[0].get_legend_handles_labels(), loc='lower center', ncol=3)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

**Resultados**

- **El ángulo de visión es el parámetro más influyente en la cohesión:** con 240° el grupo mayor sube 32 puntos y con 360° baja 23. Es justo el parámetro sin valor publicado (supuesto de 300°), así que las conclusiones sobre la cohesión dependen del tamaño real del punto ciego.
- **La recuperación media domina la energía:** ±20% cambia la energía media en −0,39 y +0,33. También cambia la distancia al vecino, porque las aves cansadas vuelan más lento y la parvada se estira.
- **`inner_radius` controla la distancia al vecino** (±0,12 a 0,15 m), como en la calibración de la sección 4.1.
- **El factor de carga máximo afecta la energía** (+0,17 y −0,12): permitir giros más cerrados aumenta el gasto por giro.
- **La separación, el peso de los líderes, `e_crit` y la proporción de juveniles** tienen efectos pequeños, la mayoría dentro del intervalo de la base.
- **La precisión de rumbo varía tanto entre réplicas** (intervalo de la base ±0,24) que ningún cambio de ±20% la mueve de forma distinguible.

## 7. Validación

La validación tiene tres partes: comprobar que el código hace lo que se diseñó (verificación), comparar los resultados con datos reales y hechos estilizados, y revisar que el comportamiento se vea razonable (validez aparente).

### 7.1 Verificación del código

Pruebas automáticas de cada mecanismo. La celda falla si alguna no pasa.

In [ ]:
pruebas = []

def prueba(grupo, nombre, condicion):
    """ Registra el resultado de una prueba. """
    pruebas.append({'grupo': grupo, 'prueba': nombre, 'resultado': 'pasa' if condicion else 'FALLA'})


def kpis_de(Modelo, parametros, pasos=200):
    """ KPIs por paso de una corrida corta. """
    return Modelo(dict(parametros, steps=pasos)).run(display=False).variables[Modelo.__name__]


# 1. Geometría del torus
d = desplazamiento_toroidal(np.array([1., 1.]), np.array([199., 199.]), 200)
prueba('Torus', 'El desplazamiento más corto cruza el borde', np.allclose(d, [-2, -2]))
m = ModeloReferencia(dict(parametros_ref, population=2))
m.sim_setup()
a, b = m.agents
m.space.move_to(a, np.array([0.5, 100.]))
m.space.move_to(b, np.array([199.5, 100.]))
prueba('Torus', 'Un vecino al otro lado del borde se detecta', len(a.space.neighbors(a, distance=2)) == 1)

# 2. Cada etapa, con sus parámetros nuevos por defecto, reproduce exactamente a la anterior
for Modelo, Anterior, p in [(ModeloPercepcionKPI, ModeloReferenciaKPI, parametros_ref),
                            (ModeloRolesKPI, ModeloPercepcionKPI, parametros_percepcion),
                            (ModeloEnergiaKPI, ModeloRolesKPI, parametros_roles)]:
    nueva, previa = kpis_de(Modelo, p), kpis_de(Anterior, p)
    prueba('Equivalencia', f'{Modelo.__name__[:-3]} con valores por defecto = {Anterior.__name__[:-3]}',
           nueva[previa.columns].equals(previa))

# 3. Percepción: los k vecinos visibles coinciden con un cálculo de fuerza bruta
p = parametros_percepcion
m = ModeloPercepcion(p)
m.sim_setup()
for _ in range(100):
    m.sim_step()
pos = np.array(list(m.space.positions.values()))
correctos = 0
for ave in m.space.positions:
    disp = desplazamiento_toroidal(ave.pos, pos, p['size'])
    dist = np.linalg.norm(disp, axis=1)
    visible = ((dist > 0) & (dist <= p['vision_radius'])
               & (disp @ ave.heading / np.maximum(dist, 1e-12) >= np.cos(np.radians(p['vision_angle'] / 2))))
    cand = np.where(visible)[0]
    esperados = np.sort(dist[cand[np.argsort(dist[cand])[:p['n_neighbors']]]])
    obtenidos = np.sort(np.linalg.norm(ave.vecinos()[0], axis=1))
    correctos += np.allclose(esperados, obtenidos)
prueba('Percepción', f'Los {p["n_neighbors"]} vecinos visibles coinciden con fuerza bruta en todas las aves', correctos == len(pos))

for angulo, esperado in [(360, 1), (300, 0)]:
    m = ModeloPercepcion(dict(p, population=2, vision_angle=angulo))
    m.sim_setup()
    a, b = m.agents
    a.heading = np.array([1., 0.])
    m.space.move_to(a, np.array([100., 100.]))
    m.space.move_to(b, np.array([98., 100.]))
    prueba('Percepción', f'Un ave 2 m detrás se percibe con {angulo}°: {"sí" if esperado else "no"}', len(a.vecinos()[0]) == esperado)

# 4. Roles: proporciones exactas y parámetros por rol
for n, fl, fj in [(200, 0.10, 0.45), (7, 1 / 3, 1 / 3), (150, 0.02, 0.5)]:
    m = ModeloRoles(dict(parametros_roles, population=n, frac_lider=fl, frac_juvenil=fj))
    m.sim_setup()
    conteo = {r: sum(a.role == r for a in m.agents) for r in ROLES}
    prueba('Roles', f'{n} aves con {fl:.0%} líderes y {fj:.0%} juveniles: conteo exacto que suma {n}',
           conteo == m.conteo_roles and sum(conteo.values()) == n and abs(conteo['lider'] - fl * n) < 1)
m = ModeloRoles(dict(parametros_roles, alignment_strength__juvenil=0.1))
m.sim_setup()
prueba('Roles', 'Un parámetro con sufijo __juvenil cambia solo a los juveniles',
       all((a.p.alignment_strength == 0.1) == (a.role == 'juvenil') for a in m.agents))

# 5. Funciones de costo
prueba('Energía', 'Costo por rapidez: 1 a 10 m/s, 0,835 a 6,3 m/s y 1,196 a 14,4 m/s',
       np.allclose([costo_rapidez(10), costo_rapidez(6.3), costo_rapidez(14.4)], [1, 0.835, 1.196], atol=1e-3))
prueba('Energía', 'Costo por giro: 1 en vuelo recto y 4 con un giro de 2 g',
       costo_giro(10, 0, 0.1) == 1 and np.isclose(costo_giro(10, G * np.sqrt(3) * 0.01 / 1.0, 0.1), 4))
prueba('Energía', 'Racha: un giro aislado no se encarece y el multiplicador se satura en streak_max',
       costo_racha(1, 2, 5) == 1 and np.isclose(costo_racha(200, 2, 5), 2))
m = ModeloEnergia(parametros_energia)
m.sim_setup()
r = np.array([a.recovery_rate for a in m.agents])
prueba('Energía', 'Recuperación individual con la media y la variación indicadas (±5% y ±0,03)',
       abs(r.mean() / parametros_energia['recovery_mean'] - 1) < 0.05 and abs(r.std() / r.mean() - parametros_energia['recovery_cv']) < 0.03)

# 6. Invariantes del modelo completo durante 500 pasos
m = ModeloEnergia(dict(parametros_energia, steps=500))
m.sim_setup()
a_max = G * np.sqrt(m.p.max_load_factor ** 2 - 1)
ok_giro = ok_rapidez = ok_energia = ok_pos = ok_rumbo = True
while m.running:
    antes = {a: (a.speed, a.v_max_actual()) for a in m.agents}
    m.sim_step()
    for a in m.agents:
        v_prev, v_max_prev = antes[a]
        ok_giro &= abs(a.turn) <= a_max * m.p.dt ** 2 / v_prev + 1e-12
        ok_rapidez &= m.p.v_min - 1e-12 <= a.speed <= v_max_prev + 1e-12
        ok_energia &= 0 <= a.energy <= 1
        ok_rumbo &= np.isclose(np.linalg.norm(a.heading), 1)
    pos = np.array(list(m.space.positions.values()))
    ok_pos &= bool((pos >= 0).all() and (pos < m.p.size).all())
prueba('Invariantes', 'El giro por paso nunca supera el máximo para la rapidez del ave', ok_giro)
prueba('Invariantes', 'La rapidez queda entre v_min y la rapidez máxima que permite la energía', ok_rapidez)
prueba('Invariantes', 'La energía queda entre 0 y 1', ok_energia)
prueba('Invariantes', 'El rumbo siempre mide 1', ok_rumbo)
prueba('Invariantes', 'Las posiciones quedan dentro del torus', ok_pos)

# 7. Reproducibilidad del modelo completo
def posiciones_finales(semilla):
    m = ModeloEnergia(dict(parametros_energia, steps=200, seed=semilla))
    m.run(display=False)
    return np.array(list(m.space.positions.values()))
prueba('Reproducibilidad', 'La misma semilla da exactamente el mismo resultado', np.array_equal(posiciones_finales(1), posiciones_finales(1)))
prueba('Reproducibilidad', 'Una semilla distinta da otro resultado', not np.array_equal(posiciones_finales(1), posiciones_finales(2)))

resultado_pruebas = pd.DataFrame(pruebas)
assert (resultado_pruebas.resultado == 'pasa').all(), 'Hay pruebas que fallan'
resultado_pruebas

### 7.2 Comparación con datos reales y hechos estilizados

In [ ]:
def valor(config, kpi):
    m, ic = media_ic(e1[e1.config == config], 'config', [kpi])
    return f'{m.iloc[0, 0]:.3f} ± {ic.iloc[0, 0]:.3f}'

comparacion = pd.DataFrame([
    ['Distancia al vecino (m)', '0,68 a 1,51', 'Ballerini et al. (2008b)', valor('Referencia', 'dist_vecino'), valor('+ Energía (completo)', 'dist_vecino')],
    ['Rapidez de la parvada (m/s)', '6,9 a 15,2 (media 10,6)', 'Ballerini et al. (2008b)', valor('Referencia', 'rapidez_media'), valor('+ Energía (completo)', 'rapidez_media')],
    ['Polarización', '0,96 ± 0,03', 'Cavagna et al. (2010)', valor('Referencia', 'polarizacion'), valor('+ Energía (completo)', 'polarizacion')],
    ['Aves a menos de 0,38 m', 'casi ninguna', 'Ballerini et al. (2008b)', valor('Referencia', 'frac_colision'), valor('+ Energía (completo)', 'frac_colision')],
    ['Juveniles fatigados / adultos fatigados', 'los juveniles gastan más', 'Rotics et al. (2016)', '—',
     f"{e1[e1.config == '+ Energía (completo)'].final_fatigada_juvenil.mean():.2f} / {e1[e1.config == '+ Energía (completo)'].final_fatigada_adulto.mean():.2f}"],
], columns=['Indicador', 'Valor observado', 'Fuente', 'Referencia', 'Modelo completo'])
comparacion

**Coincidencias**
- **Rapidez:** el modelo completo vuela a ~9,7 m/s en promedio, dentro del rango observado y cerca de la media de 10,6 m/s.
- **Polarización:** el modelo completo (0,95) queda dentro del rango observado (0,96 ± 0,03), mientras que la referencia (0,995) queda por encima. Las diferencias entre aves (roles y energía) agregan el desorden que la referencia no tiene.
- **Colisiones:** casi ninguna ave queda a menos de 0,38 m de otra, en ambos modelos.
- **Juveniles:** se fatigan tres veces más que los adultos (66% frente a 22%) y se rezagan más, como reportan Rotics et al. (2016) para cigüeñas.
- **Liderazgo:** la precisión de rumbo aumenta con la fracción de líderes (E3), como en Couzin et al. (2005).

**Diferencias**
- **Distancia al vecino:** la referencia (1,06 m) coincide con la mediana observada, pero el modelo completo (1,85 m) queda por encima del rango. Las diferencias de rapidez estiran la parvada. Con una cohesión más fuerte (0,02, E4) la distancia vuelve al rango, a costa de más fatiga: una recalibración de la cohesión en el modelo completo es un ajuste pendiente.
- **Cohesión con interacción topológica:** Ballerini et al. (2008a) argumentan que la interacción topológica da parvadas más cohesionadas que la métrica frente a cambios de densidad. En este modelo, la percepción métrica de la referencia es más cohesionada (100% frente a 94% en el grupo mayor), y la topológica no logra formar una parvada desde el desorden (sección 3.1). La comparación no es directa: aquí no se aplicaron perturbaciones de densidad, como el ataque de un depredador.
- **Fragmentación con líderes:** con percepción realista, los líderes dividen la parvada en grupos paralelos (secciones 3.2 y 6.3), algo que Couzin et al. (2005) no reportan con su modelo métrico.

### 7.3 Validez aparente

- **Parvada formada:** en la referencia y en la Etapa 1, la parvada se mantiene como un grupo compacto, alineado y elongado (sección 3.0), con proporciones parecidas a las observadas.
- **Modelo completo:** la figura de la sección 3.4 muestra franjas curvas de aves alineadas, varios grupos que vuelan en direcciones parecidas y aves cansadas (más transparentes) rezagadas detrás de los grupos. Es un patrón parecido al de las murmuraciones de estornino, que cambian de forma y se dividen y reúnen.
- **Comportamientos emergentes** que no se programaron directamente: los líderes gastan más energía porque aceleran (sección 3.3); la parvada espera a las aves cansadas solo si la cohesión es fuerte (E4); y el punto ciego favorece la cohesión (E2).

## 8. Conclusiones y limitaciones

- Hallazgos principales.
- Qué quedó fuera y por qué.
- Cómo se podría extender.

## 9. Referencias

- Ballerini, M., Cabibbo, N., Candelier, R., Cavagna, A., Cisbani, E., Giardina, I., Lecomte, V., Orlandi, A., Parisi, G., Procaccini, A., Viale, M. y Zdravkovic, V. (2008a). Interaction ruling animal collective behavior depends on topological rather than metric distance: evidence from a field study. *Proceedings of the National Academy of Sciences*, 105(4), 1232–1237. https://doi.org/10.1073/pnas.0711437105
- Ballerini, M., Cabibbo, N., Candelier, R., Cavagna, A., Cisbani, E., Giardina, I., Orlandi, A., Parisi, G., Procaccini, A., Viale, M. y Zdravkovic, V. (2008b). Empirical investigation of starling flocks: a benchmark study in collective animal behaviour. *Animal Behaviour*, 76(1), 201–215. https://arxiv.org/abs/0802.1667
- Bateson, M. y Asher, L. (2010). The European starling. En R. Hubrecht y J. Kirkwood (Eds.), *The UFAW Handbook on the Care and Management of Laboratory and Other Research Animals* (8.ª ed., pp. 697–705). Wiley-Blackwell. https://www.staff.ncl.ac.uk/melissa.bateson/Bateson_Asher_2010.pdf
- BTO, British Trust for Ornithology (s. f.). Starling: BirdFacts. https://ociweb.bto.org/learn/about-birds/birdfacts/starling
- Cavagna, A., Cimarelli, A., Giardina, I., Parisi, G., Santagati, R., Stefanini, F. y Viale, M. (2010). Scale-free correlations in starling flocks. *Proceedings of the National Academy of Sciences*, 107(26), 11865–11870. https://doi.org/10.1073/pnas.1005766107
- Couzin, I. D., Krause, J., Franks, N. R. y Levin, S. A. (2005). Effective leadership and decision-making in animal groups on the move. *Nature*, 433, 513–516. https://doi.org/10.1038/nature03236
- Foramitti, J. (2021). AgentPy: A package for agent-based modeling in Python. *Journal of Open Source Software*, 6(62), 3065. https://doi.org/10.21105/joss.03065
- Pomeroy, H. y Heppner, F. (1977). Laboratory determination of startle reaction time of the starling (*Sturnus vulgaris*). *Animal Behaviour*, 25, 720–725. https://doi.org/10.1016/0003-3472(77)90121-X
- Reynolds, C. W. (1987). Flocks, herds and schools: A distributed behavioral model. *ACM SIGGRAPH Computer Graphics*, 21(4), 25–34. https://doi.org/10.1145/37402.37406
- Rotics, S., Kaatz, M., Resheff, Y. S., Turjeman, S. F., Zurell, D., Sapir, N., Eggers, U., Flack, A., Fiedler, W., Jeltsch, F., Wikelski, M. y Nathan, R. (2016). The challenges of the first migration: movement and behaviour of juvenile vs. adult white storks with insights regarding juvenile mortality. *Journal of Animal Ecology*, 85(4), 938–947. https://doi.org/10.1111/1365-2656.12525
- Tisdale, V. y Fernández-Juricic, E. (2009). Vigilance and predator detection vary between avian species with different visual acuity and coverage. *Behavioral Ecology*, 20(5), 936–945. https://ideas.repec.org/a/oup/beheco/v20y2009i5p936-945.html
- Usherwood, J. R., Stavrou, M., Lowe, J. C., Roskilly, K. y Wilson, A. M. (2011). Flying in a flock comes at a cost in pigeons. *Nature*, 474, 494–497. https://doi.org/10.1038/nature10164
- Versteegh, M. A., Helm, B., Dingemanse, N. J. y Tieleman, B. I. (2008). Repeatability and individual correlates of basal metabolic rate and total evaporative water loss in birds: a case study in European stonechats. *Comparative Biochemistry and Physiology Part A*, 150(4), 452–457. https://doi.org/10.1016/j.cbpa.2008.05.006
- Ward, S., Möller, U., Rayner, J. M. V., Jackson, D. M., Bilo, D., Nachtigall, W. y Speakman, J. R. (2001). Metabolic power, mechanical power and efficiency during wind tunnel flight by the European starling *Sturnus vulgaris*. *Journal of Experimental Biology*, 204(19), 3311. https://cob.silverchair.com/jeb/article-pdf/204/19/3311/1238626/3311.pdf
- Wilson, R. P. et al. (2013). Turn costs change the value of animal search paths. *Ecology Letters*, 16(9), 1145–1150. https://doi.org/10.1111/ele.12149